# Load data

1. load the following dataset from data folder
    - modeling_data.csv
    - has_2025_data.csv
    - full_period_data.csv
2. load numerical and categorical data from model_feature_list.json

In [ ]:
from pathlib import Path
import json, pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
modeling_data = pd.read_csv(ROOT / "data" / "modeling_data.csv")
has_2025_data = pd.read_csv(ROOT / "data" / "has_2025_data.csv")
full_period_data = pd.read_csv(ROOT / "data" / "full_period_data.csv")
with open(ROOT / "data" / "model_feature_lists.json", encoding="utf-8") as file:
    model_features = json.load(file)
numerical_columns, categorical_columns = model_features["numerical_columns"], model_features["categorical_columns"]

# Latest-period Calibration

## Baseline Linear-Regression

### Step 1 — Define feature families

Group predictors by their economic meaning. The three market families are used to measure model breadth. Geographic variables are tracked separately and do not increase the market-family coverage score.

In [ ]:
from itertools import combinations
import time

import numpy as np
import statsmodels.api as sm
from scipy.stats import spearmanr
from IPython.display import display

TARGET_COLUMN = "Market_Score"
GROUP_COLUMN = "Zip Code"

feature_families = {
    "price": [
        "MEDIAN_SALE_PRICE", "MEDIAN_SALE_PRICE_MOM", "MEDIAN_SALE_PRICE_YOY",
        "MEDIAN_LIST_PRICE", "MEDIAN_LIST_PRICE_MOM", "MEDIAN_LIST_PRICE_YOY",
        "MEDIAN_PPSF", "MEDIAN_PPSF_MOM", "MEDIAN_PPSF_YOY",
        "MEDIAN_LIST_PPSF", "MEDIAN_LIST_PPSF_MOM", "MEDIAN_LIST_PPSF_YOY",
    ],
    "demand_supply": [
        "HOMES_SOLD", "HOMES_SOLD_MOM", "HOMES_SOLD_YOY",
        "PENDING_SALES", "PENDING_SALES_MOM", "PENDING_SALES_YOY",
        "NEW_LISTINGS", "NEW_LISTINGS_MOM", "NEW_LISTINGS_YOY",
        "INVENTORY", "INVENTORY_MOM", "INVENTORY_YOY",
    ],
    "market_speed": [
        "MEDIAN_DOM", "MEDIAN_DOM_MOM", "MEDIAN_DOM_YOY",
        "AVG_SALE_TO_LIST", "AVG_SALE_TO_LIST_MOM", "AVG_SALE_TO_LIST_YOY",
        "SOLD_ABOVE_LIST", "SOLD_ABOVE_LIST_MOM", "SOLD_ABOVE_LIST_YOY",
        "OFF_MARKET_IN_TWO_WEEKS", "OFF_MARKET_IN_TWO_WEEKS_MOM",
        "OFF_MARKET_IN_TWO_WEEKS_YOY",
    ],
    "geography": ["County", "PARENT_METRO_REGION"],
}

feature_to_family = {
    feature: family
    for family, features in feature_families.items()
    for feature in features
}

candidate_variables = list(dict.fromkeys(numerical_columns + categorical_columns))
missing_candidate_variables = sorted(set(candidate_variables) - set(modeling_data.columns))
if missing_candidate_variables:
    raise ValueError(f"Candidate variables missing from modeling_data: {missing_candidate_variables}")

unclassified_variables = sorted(set(candidate_variables) - set(feature_to_family))
if unclassified_variables:
    raise ValueError(f"Candidate variables missing a feature family: {unclassified_variables}")

pd.DataFrame({
    "Feature family": feature_families.keys(),
    "Number of variables": [len(values) for values in feature_families.values()],
})

### Step 2 — Define incompatible feature groups

Exclude combinations containing more than one level, MoM, or YoY version of the same underlying metric. County and parent metro are also mutually exclusive because they are overlapping geographic representations.

In [ ]:
incompatible_groups = [
    ["MEDIAN_SALE_PRICE", "MEDIAN_SALE_PRICE_MOM", "MEDIAN_SALE_PRICE_YOY"],
    ["MEDIAN_LIST_PRICE", "MEDIAN_LIST_PRICE_MOM", "MEDIAN_LIST_PRICE_YOY"],
    ["MEDIAN_PPSF", "MEDIAN_PPSF_MOM", "MEDIAN_PPSF_YOY"],
    ["MEDIAN_LIST_PPSF", "MEDIAN_LIST_PPSF_MOM", "MEDIAN_LIST_PPSF_YOY"],
    ["HOMES_SOLD", "HOMES_SOLD_MOM", "HOMES_SOLD_YOY"],
    ["PENDING_SALES", "PENDING_SALES_MOM", "PENDING_SALES_YOY"],
    ["NEW_LISTINGS", "NEW_LISTINGS_MOM", "NEW_LISTINGS_YOY"],
    ["INVENTORY", "INVENTORY_MOM", "INVENTORY_YOY"],
    ["MEDIAN_DOM", "MEDIAN_DOM_MOM", "MEDIAN_DOM_YOY"],
    ["AVG_SALE_TO_LIST", "AVG_SALE_TO_LIST_MOM", "AVG_SALE_TO_LIST_YOY"],
    ["SOLD_ABOVE_LIST", "SOLD_ABOVE_LIST_MOM", "SOLD_ABOVE_LIST_YOY"],
    [
        "OFF_MARKET_IN_TWO_WEEKS", "OFF_MARKET_IN_TWO_WEEKS_MOM",
        "OFF_MARKET_IN_TWO_WEEKS_YOY",
    ],
    ["County", "PARENT_METRO_REGION"],
]
incompatible_sets = [set(group) for group in incompatible_groups]

def is_valid_combination(variables):
    selected = set(variables)
    return all(len(selected.intersection(group)) <= 1 for group in incompatible_sets)

pd.DataFrame({
    "Incompatible group": range(1, len(incompatible_groups) + 1),
    "Variables": [", ".join(group) for group in incompatible_groups],
})

### Step 3 — Build regression design matrices

Numerical variables enter directly. Nominal categorical variables are one-hot encoded with one reference level omitted. Each categorical field remains one original model variable even though it creates multiple coefficients.

In [ ]:
def build_design_matrix(data, variables):
    parts = []
    coefficient_blocks = {}

    for variable in variables:
        if variable in categorical_columns:
            values = data[variable].astype("string").fillna("__MISSING__")
            encoded = pd.get_dummies(
                values,
                prefix=variable,
                prefix_sep="=",
                drop_first=True,
                dtype=float,
            )
            if encoded.shape[1] == 0:
                raise ValueError(f"Categorical variable {variable} has fewer than two levels.")
            parts.append(encoded)
            coefficient_blocks[variable] = encoded.columns.tolist()
        else:
            numeric = pd.to_numeric(data[variable], errors="coerce").astype(float)
            parts.append(numeric.rename(variable).to_frame())
            coefficient_blocks[variable] = [variable]

    design = pd.concat(parts, axis=1)
    if design.isna().any().any():
        missing = design.columns[design.isna().any()].tolist()
        raise ValueError(f"Missing values in design matrix: {missing}")
    if not np.isfinite(design.to_numpy()).all():
        raise ValueError("The design matrix contains infinite values.")

    # design = sm.add_constant(design, has_constant="add").astype(float)
    design = design.astype(float)
    design.insert(0, "const", 1.0)
    return design, coefficient_blocks


sample_design, sample_blocks = build_design_matrix(
    modeling_data, candidate_variables[:2]
)
display(sample_design.head())
display(pd.Series(sample_blocks, name="Coefficient columns"))

### Step 4 — Define expected coefficient signs

Sign expectations assume that a higher Market Score represents a stronger market. Price levels and new listings remain unconstrained because their direction is not unambiguous. Categorical effects depend on the omitted reference group, so sign testing does not apply to them.

In [ ]:
def expected_sign(variable):
    if variable in categorical_columns:
        return "not_applicable", "Nominal category coefficients depend on the reference level."
    if variable.startswith("MEDIAN_DOM"):
        return "negative", "Longer or increasing time on market indicates weaker market speed."
    if variable.startswith("INVENTORY"):
        return "negative", "Higher or increasing inventory generally indicates weaker seller conditions."
    if variable.startswith(("AVG_SALE_TO_LIST", "SOLD_ABOVE_LIST", "OFF_MARKET_IN_TWO_WEEKS")):
        return "positive", "Higher competition and faster absorption indicate a stronger market."
    if variable.startswith(("HOMES_SOLD", "PENDING_SALES")):
        return "positive", "Higher or increasing transaction activity indicates stronger demand."
    if variable.startswith("NEW_LISTINGS"):
        return "unconstrained", "New listings reflect both supply and seller confidence."
    if variable.endswith(("_MOM", "_YOY")) and variable.startswith(
        ("MEDIAN_SALE_PRICE", "MEDIAN_LIST_PRICE", "MEDIAN_PPSF", "MEDIAN_LIST_PPSF")
    ):
        return "positive", "Positive price momentum is provisionally associated with market strength."
    if variable.startswith(("MEDIAN_SALE_PRICE", "MEDIAN_LIST_PRICE", "MEDIAN_PPSF", "MEDIAN_LIST_PPSF")):
        return "unconstrained", "Price level measures market value, not necessarily current strength."
    return "unconstrained", "No unambiguous economic direction was assigned."


feature_sign_lookup = pd.DataFrame(
    [
        {
            "Variable": variable,
            "Expected_sign": expected_sign(variable)[0],
            "Rationale": expected_sign(variable)[1],
        }
        for variable in candidate_variables
    ]
)
display(feature_sign_lookup)

### Step 5 — Generate all valid 2–4 variable combinations

Generate the exhaustive candidate set and remove combinations that violate an incompatibility rule. `MODEL_LIMIT` can be set temporarily for testing; leave it as `None` for the complete search.

In [ ]:
all_combinations = [
    combination
    for size in range(3, 4)  # number of variables to be included
    for combination in combinations(candidate_variables, size)
]
valid_combinations = [
    combination for combination in all_combinations if is_valid_combination(combination)
]
excluded_combination_count = len(all_combinations) - len(valid_combinations)

MODEL_LIMIT = None
combinations_to_fit = (
    valid_combinations if MODEL_LIMIT is None else valid_combinations[:MODEL_LIMIT]
)

combination_summary = pd.DataFrame({
    "Measure": ["All combinations", "Excluded by rules", "Valid combinations", "Scheduled to fit"],
    "Count": [
        len(all_combinations), excluded_combination_count,
        len(valid_combinations), len(combinations_to_fit),
    ],
})
display(combination_summary)

### Step 6 — Define variable-level significance tests

For numerical variables, report the coefficient t-statistic and p-value. For categorical variables, test the complete dummy block jointly with a partial F-test rather than selecting one category’s p-value.

In [ ]:
def variable_significance_tests(model, coefficient_blocks, variables):
    tests = {}
    parameter_names = model.params.index.tolist()

    for variable in variables:
        block = coefficient_blocks[variable]
        if variable in categorical_columns:
            restriction = np.zeros((len(block), len(parameter_names)))
            for row, coefficient in enumerate(block):
                restriction[row, parameter_names.index(coefficient)] = 1.0
            joint_test = model.f_test(restriction)
            tests[variable] = {
                "test_type": "joint_F",
                "test_statistic": float(np.asarray(joint_test.fvalue).squeeze()),
                "p_value": float(np.asarray(joint_test.pvalue).squeeze()),
            }
        else:
            tests[variable] = {
                "test_type": "t",
                "test_statistic": float(model.tvalues[variable]),
                "p_value": float(model.pvalues[variable]),
            }

    return tests

### Step 7 — Calculate model statistics and diagnostics

Fit each candidate with OLS on all modeling records. Record goodness of fit, error, information criteria, overall significance, multicollinearity, influence, and coefficient-sign consistency. Because cross-validation was excluded, `RMSE` and `MAE` are in-sample measures.

In [ ]:
def calculate_vif(design):
    predictors = design.drop(columns="const", errors="ignore")
    if predictors.shape[1] == 1:
        return 1.0, 1.0

    standard_deviations = predictors.std(ddof=0)
    if standard_deviations.eq(0).any():
        raise ValueError("A design-matrix column has zero variance.")
    standardized = (predictors - predictors.mean()) / standard_deviations
    correlation = np.corrcoef(standardized.to_numpy(), rowvar=False)
    inverse_correlation = np.linalg.inv(correlation)
    vif_values = np.diag(inverse_correlation)
    return float(np.max(vif_values)), float(np.mean(vif_values))


def evaluate_signs(model, variables):
    constrained = []
    failures = []
    lookup = feature_sign_lookup.set_index("Variable")["Expected_sign"]

    for variable in variables:
        expectation = lookup[variable]
        if expectation not in {"positive", "negative"}:
            continue
        constrained.append(variable)
        coefficient = float(model.params[variable])
        if (expectation == "positive" and coefficient <= 0) or (
            expectation == "negative" and coefficient >= 0
        ):
            failures.append(variable)

    if not constrained:
        return "NOT_APPLICABLE", ""
    if failures:
        return "FAIL", ", ".join(failures)
    return "PASS", ""


def fit_candidate(variables, model_id):
    result = {
        "model_id": model_id,
        "model_status": "INVALID",
        "exclusion_reason": "",
    }
    for position in range(1, 5):
        result[f"variable_{position}"] = variables[position - 1] if position <= len(variables) else np.nan
        result[f"test_type_{position}"] = np.nan
        result[f"test_statistic_{position}"] = np.nan
        result[f"p_value_{position}"] = np.nan

    try:
        design, coefficient_blocks = build_design_matrix(modeling_data, variables)
        target = pd.to_numeric(modeling_data[TARGET_COLUMN], errors="raise").astype(float)
        if np.linalg.matrix_rank(design.to_numpy()) < design.shape[1]:
            raise ValueError("Rank-deficient design matrix.")

        model = sm.OLS(target, design).fit()
        predictions = model.predict(design)
        residuals = target - predictions
        variable_tests = variable_significance_tests(model, coefficient_blocks, variables)
        max_vif, mean_vif = calculate_vif(design)
        influence = model.get_influence()
        cooks_distance = influence.cooks_distance[0]
        sign_check, sign_failures = evaluate_signs(model, variables)

        n_observations = int(model.nobs)
        n_parameters = int(len(model.params))
        aicc_denominator = n_observations - n_parameters - 1
        aicc = (
            float(model.aic + (2 * n_parameters * (n_parameters + 1)) / aicc_denominator)
            if aicc_denominator > 0 else np.nan
        )

        for position, variable in enumerate(variables, start=1):
            test = variable_tests[variable]
            result[f"test_type_{position}"] = test["test_type"]
            result[f"test_statistic_{position}"] = test["test_statistic"]
            result[f"p_value_{position}"] = test["p_value"]

        result.update({
            "r_squared": float(model.rsquared),
            "adjusted_r_squared": float(model.rsquared_adj),
            "RMSE": float(np.sqrt(np.mean(np.square(residuals)))),
            "MAE": float(np.mean(np.abs(residuals))),
            "spearman": float(spearmanr(target, predictions)[0]),
            "f_statistics": float(model.fvalue),
            "f_p_value": float(model.f_pvalue),
            "aic": float(model.aic),
            "aicc": aicc,
            "bic": float(model.bic),
            "vif": max_vif,
            "max_vif": max_vif,
            "mean_vif": mean_vif,
            "condition_number": float(model.condition_number),
            "max_cooks_distance": float(np.max(cooks_distance)),
            "sign_check": sign_check,
            "sign_failures": sign_failures,
            "n_observations": n_observations,
            "n_original_variables": len(variables),
            "n_parameters": n_parameters,
            "model_status": "VALID",
        })
    except Exception as error:
        result["exclusion_reason"] = f"{type(error).__name__}: {error}"

    return result

### Step 8 — Calculate feature-family coverage

Record whether a candidate includes price, demand/supply, and market-speed information. Coverage ranges from zero to three and acts as a secondary model-selection criterion, not a substitute for fit and diagnostics.

In [ ]:
market_families = ["price", "demand_supply", "market_speed"]

def add_family_coverage(result, variables):
    included = {feature_to_family[variable] for variable in variables}
    for family in market_families:
        result[f"{family}_included"] = int(family in included)
    result["family_coverage"] = sum(family in included for family in market_families)
    result["families_included"] = ", ".join(
        family for family in feature_families if family in included
    )
    return result


display(pd.DataFrame([
    add_family_coverage({}, combination)
    for combination in combinations_to_fit[:5]
]))

### Step 9 — Run the exhaustive model search

Fit every scheduled candidate and retain both successful and invalid results. The full search may take substantial time. Progress is printed every 1,000 models. Leave `MODEL_LIMIT=None` in Step 5 to run all valid combinations.

In [ ]:
search_start_time = time.perf_counter()
model_result_rows = []

for model_number, variables in enumerate(combinations_to_fit, start=1):
    fitted_result = fit_candidate(variables, model_id=model_number)
    model_result_rows.append(add_family_coverage(fitted_result, variables))
    if model_number % 1000 == 0 or model_number == len(combinations_to_fit):
        elapsed_minutes = (time.perf_counter() - search_start_time) / 60
        print(
            f"Processed {model_number:,}/{len(combinations_to_fit):,} models "
            f"in {elapsed_minutes:,.1f} minutes"
        )

model_results = pd.DataFrame(model_result_rows)
valid_model_results = model_results.loc[
    model_results["model_status"].eq("VALID")
].copy()

run_summary = pd.DataFrame({
    "Measure": [
        "All generated combinations", "Excluded by incompatibility rules",
        "Models attempted", "Valid models", "Invalid models", "Elapsed minutes",
    ],
    "Value": [
        len(all_combinations), excluded_combination_count,
        len(model_results), len(valid_model_results),
        model_results["model_status"].ne("VALID").sum(),
        (time.perf_counter() - search_start_time) / 60,
    ],
})
display(run_summary)

### Step 10 — Rank valid models

Rank models using in-sample RMSE first, followed by rank correlation, sign consistency, multicollinearity, feature-family coverage, and parsimony. These rankings are exploratory because exhaustive selection makes conventional significance tests optimistic.

In [ ]:
model_results.head()
# model_results.model_status.unique()
model_results.exclusion_reason.unique()

In [ ]:
if valid_model_results.empty:
    raise RuntimeError("No valid candidate models were fitted.")

valid_model_results["sign_rank"] = valid_model_results["sign_check"].map({
    "PASS": 0,
    "NOT_APPLICABLE": 1,
    "FAIL": 2,
}).fillna(3)
valid_model_results["vif_rank"] = (~valid_model_results["max_vif"].le(5)).astype(int)

ranked_model_results = (
    valid_model_results.sort_values(
        [
            "RMSE", "spearman", "sign_rank", "vif_rank",
            "family_coverage", "n_parameters",
        ],
        ascending=[True, False, True, True, False, True],
    )
    .reset_index(drop=True)
)
ranked_model_results.insert(0, "model_rank", np.arange(1, len(ranked_model_results) + 1))

display(ranked_model_results.head(20))
display(
    ranked_model_results["family_coverage"]
    .value_counts()
    .sort_index()
    .rename_axis("Family coverage")
    .to_frame("Number of models")
)

### Step 11 — Save and summarize results

Save the complete ranked valid-model table, invalid-model diagnostics, and the expected-sign lookup under `outputs/model_search`. Display the best model’s variables, tests, and principal diagnostics.

The reported p-values, t-statistics, and F-statistics are descriptive after an exhaustive search. They should not be interpreted as confirmatory inference without independent validation.

In [ ]:
from IPython.display import display, Markdown

output_dir = ROOT / "outputs" / "model_search"
output_dir.mkdir(parents=True, exist_ok=True)

ranked_model_results.to_csv(output_dir / "model_search_results.csv", index=False)
model_results.loc[model_results["model_status"].ne("VALID")].to_csv(
    output_dir / "invalid_model_results.csv", index=False
)
feature_sign_lookup.to_csv(output_dir / "feature_sign_lookup.csv", index=False)

best_model = ranked_model_results.iloc[0]
best_model_variables = [
    best_model[f"variable_{position}"]
    for position in range(1, 5)
    if pd.notna(best_model[f"variable_{position}"])
]
best_design, best_blocks = build_design_matrix(modeling_data, best_model_variables)
best_fitted_model = sm.OLS(
    pd.to_numeric(modeling_data[TARGET_COLUMN]).astype(float), best_design
).fit()

display(Markdown("#### Best model variables and variable-level tests"))
display(pd.DataFrame({
    "Variable": best_model_variables,
    "Test type": [best_model[f"test_type_{i}"] for i in range(1, len(best_model_variables) + 1)],
    "Test statistic": [best_model[f"test_statistic_{i}"] for i in range(1, len(best_model_variables) + 1)],
    "P-value": [best_model[f"p_value_{i}"] for i in range(1, len(best_model_variables) + 1)],
}))

display(Markdown("#### Best model coefficients"))
display(pd.DataFrame({
    "Coefficient": best_fitted_model.params.index,
    "Estimate": best_fitted_model.params.values,
    "T-statistic": best_fitted_model.tvalues.values,
    "P-value": best_fitted_model.pvalues.values,
}))

display(Markdown("#### Best model diagnostics"))
display(best_model[[
    "model_rank", "r_squared", "adjusted_r_squared", "RMSE", "MAE",
    "spearman", "f_statistics", "f_p_value", "aicc", "bic", "max_vif",
    "condition_number", "max_cooks_distance", "sign_check",
    "family_coverage", "families_included", "n_parameters",
]].to_frame("Value"))

print(f"Saved model-search outputs to: {output_dir}")

## LassoCV Regression

### Step 1 — Prepare modeling inputs and enforce incompatibility rules

Use the existing feature lists and target. Remove identifiers, flags, unavailable columns, and the target. Before fitting, retain at most one feature from every existing incompatibility group. Numerical alternatives are ranked by absolute Spearman association and categorical alternatives by univariate R-squared. The same selection is repeated using training-only data inside each outer fold to prevent leakage.


In [ ]:
from IPython.display import Markdown
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import warnings
from sklearn.exceptions import ConvergenceWarning

LASSO_RANDOM_SEED = 42
LASSO_ZERO_TOLERANCE = 1e-8
LASSO_ALPHAS = np.logspace(-4, 2, 200)

excluded_features = {TARGET_COLUMN, GROUP_COLUMN, "Zip Code", "has_2025", "latest_period"}
base_lasso_numerical_columns = [
    column for column in numerical_columns
    if column in modeling_data.columns and column not in excluded_features
]
base_lasso_categorical_columns = [
    column for column in categorical_columns
    if column in modeling_data.columns and column not in excluded_features
]


def compatibility_strength(data, variable):
    target = pd.to_numeric(data[TARGET_COLUMN], errors="coerce")
    if variable in base_lasso_categorical_columns:
        category_means = data.groupby(variable, dropna=False)[TARGET_COLUMN].transform("mean")
        valid = target.notna() & category_means.notna()
        return r2_score(target.loc[valid], category_means.loc[valid]) if valid.sum() > 1 else -np.inf
    values = pd.to_numeric(data[variable], errors="coerce")
    valid = target.notna() & values.notna()
    if valid.sum() < 3 or values.loc[valid].nunique() < 2:
        return -np.inf
    statistic = spearmanr(target.loc[valid], values.loc[valid])[0]
    return abs(float(statistic)) if np.isfinite(statistic) else -np.inf


def select_compatible_features(data, numerical_candidates, categorical_candidates):
    selected = list(numerical_candidates) + list(categorical_candidates)
    decisions = []
    for group_number, group in enumerate(incompatible_groups, start=1):
        available = [variable for variable in group if variable in selected]
        if len(available) <= 1:
            continue
        strengths = {variable: compatibility_strength(data, variable) for variable in available}
        retained = max(available, key=lambda variable: (strengths[variable], variable))
        for variable in available:
            decisions.append({
                "Incompatible_group": group_number,
                "Variable": variable,
                "Selection_strength": strengths[variable],
                "Decision": "RETAIN" if variable == retained else "EXCLUDE",
                "Retained_variable": retained,
            })
            if variable != retained:
                selected.remove(variable)
    selected_numerical = [v for v in numerical_candidates if v in selected]
    selected_categorical = [v for v in categorical_candidates if v in selected]
    return selected_numerical, selected_categorical, pd.DataFrame(decisions)


lasso_numerical_columns, lasso_categorical_columns, lasso_compatibility_decisions = (
    select_compatible_features(
        modeling_data, base_lasso_numerical_columns, base_lasso_categorical_columns
    )
)
lasso_features = lasso_numerical_columns + lasso_categorical_columns

X_lasso = modeling_data[lasso_features].copy()
y_lasso = pd.to_numeric(modeling_data[TARGET_COLUMN], errors="raise").astype(float)
zip_groups = modeling_data[GROUP_COLUMN].astype(str)
zip_repeats = zip_groups.duplicated().any()

display(lasso_compatibility_decisions)
display(pd.DataFrame({
    "Measure": ["Rows", "Unique ZIP codes", "Repeated ZIP codes", "Numerical variables", "Categorical variables"],
    "Value": [len(modeling_data), zip_groups.nunique(), zip_repeats,
              len(lasso_numerical_columns), len(lasso_categorical_columns)],
}))

### Step 2 — Build preprocessing

Median-impute and standardize numerical variables. Most-frequent-impute and one-hot encode nominal categorical variables while handling unseen validation levels. All levels are retained because the installed sklearn version does not support combining `drop="first"` with `handle_unknown="ignore"`; Lasso regularization handles the redundant dummy representation. The post-selection OLS design still uses a reference level.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(drop=None, handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(drop=None, handle_unknown="ignore", sparse=False)


def make_lasso_preprocessor(numerical_features=None, categorical_features=None):
    numerical_features = lasso_numerical_columns if numerical_features is None else numerical_features
    categorical_features = lasso_categorical_columns if categorical_features is None else categorical_features
    transformers = []
    if numerical_features:
        transformers.append(("numerical", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numerical_features))
    if categorical_features:
        transformers.append(("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder()),
        ]), categorical_features))
    return ColumnTransformer(transformers=transformers, remainder="drop")


lasso_preprocessor = make_lasso_preprocessor()

### Step 3 — Configure cross-validation

Use five grouped folds when ZIP codes repeat, preventing the same ZIP from appearing in training and validation. Otherwise, use shuffled five-fold validation.

In [ ]:
LASSO_N_SPLITS = min(5, zip_groups.nunique() if zip_repeats else len(modeling_data))
if LASSO_N_SPLITS < 2:
    raise ValueError("At least two independent groups or observations are required for cross-validation.")

if zip_repeats:
    outer_cv = GroupKFold(n_splits=LASSO_N_SPLITS)
    outer_splits = list(outer_cv.split(X_lasso, y_lasso, groups=zip_groups))
    lasso_cv_type = "GroupKFold"
else:
    outer_cv = KFold(n_splits=LASSO_N_SPLITS, shuffle=True, random_state=LASSO_RANDOM_SEED)
    outer_splits = list(outer_cv.split(X_lasso, y_lasso))
    lasso_cv_type = "KFold"

display(pd.DataFrame({
    "Setting": ["CV type", "Outer folds", "ZIP grouping used", "Random seed"],
    "Value": [lasso_cv_type, LASSO_N_SPLITS, zip_repeats, LASSO_RANDOM_SEED],
}))

### Step 4 — Tune the Lasso penalty

Fit the final LassoCV model on all modeling records after applying the incompatibility rules. Its inner folds tune alpha, which controls coefficient shrinkage and sparsity. Grouped split indices are supplied when ZIP codes repeat.


In [ ]:
def make_inner_splits(X, y, groups):
    n_groups = pd.Series(groups).nunique()
    n_splits = min(5, n_groups if zip_repeats else len(X))
    if zip_repeats:
        return list(GroupKFold(n_splits=n_splits).split(X, y, groups=groups))
    return list(KFold(n_splits=n_splits, shuffle=True,
                      random_state=LASSO_RANDOM_SEED).split(X, y))


final_inner_splits = make_inner_splits(X_lasso, y_lasso, zip_groups)
lasso_net_pipeline = Pipeline([
    ("preprocessor", make_lasso_preprocessor()),
    ("model", LassoCV(
        alphas=LASSO_ALPHAS,
        cv=final_inner_splits, max_iter=100000, tol=1e-6,
        selection="cyclic", n_jobs=-1,
    )),
])

with warnings.catch_warnings(record=True) as final_fit_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    lasso_net_pipeline.fit(X_lasso, y_lasso)

lasso_net_model = lasso_net_pipeline.named_steps["model"]
lasso_convergence_warnings = [
    str(item.message) for item in final_fit_warnings
    if issubclass(item.category, ConvergenceWarning)
]
display(pd.DataFrame({
    "Hyperparameter": ["alpha"],
    "Selected value": [lasso_net_model.alpha_],
}))

### Step 5 — Generate out-of-fold predictions

Refit preprocessing and LassoCV within every outer training fold, then predict its untouched validation fold. These out-of-fold predictions provide the primary estimates of predictive performance.

In [ ]:
lasso_oof_predictions = pd.Series(index=modeling_data.index, dtype=float)
lasso_fold_results = []
lasso_oof_warnings = []

for fold_number, (train_position, validation_position) in enumerate(outer_splits, start=1):
    training_data = modeling_data.iloc[train_position]
    fold_numerical, fold_categorical, _ = select_compatible_features(
        training_data, base_lasso_numerical_columns, base_lasso_categorical_columns
    )
    fold_features = fold_numerical + fold_categorical
    X_train = modeling_data.iloc[train_position][fold_features]
    X_validation = modeling_data.iloc[validation_position][fold_features]
    y_train = y_lasso.iloc[train_position]
    train_groups = zip_groups.iloc[train_position]
    inner_splits = make_inner_splits(X_train, y_train, train_groups)

    fold_pipeline = Pipeline([
        ("preprocessor", make_lasso_preprocessor(fold_numerical, fold_categorical)),
        ("model", LassoCV(
            alphas=LASSO_ALPHAS,
            cv=inner_splits, max_iter=100000, tol=1e-6,
            selection="cyclic", n_jobs=-1,
        )),
    ])
    with warnings.catch_warnings(record=True) as fold_warnings:
        warnings.simplefilter("always", ConvergenceWarning)
        fold_pipeline.fit(X_train, y_train)

    fold_predictions = fold_pipeline.predict(X_validation)
    lasso_oof_predictions.iloc[validation_position] = fold_predictions
    fold_model = fold_pipeline.named_steps["model"]
    lasso_fold_results.append({
        "fold": fold_number, "training_rows": len(train_position),
        "validation_rows": len(validation_position), "alpha": fold_model.alpha_,
        "compatible_features": len(fold_features),
        "RMSE": mean_squared_error(y_lasso.iloc[validation_position], fold_predictions) ** 0.5,
        "MAE": mean_absolute_error(y_lasso.iloc[validation_position], fold_predictions),
    })
    lasso_oof_warnings.extend(str(item.message) for item in fold_warnings
                                if issubclass(item.category, ConvergenceWarning))

lasso_cv_metrics = {
    "r_squared": r2_score(y_lasso, lasso_oof_predictions),
    "RMSE": mean_squared_error(y_lasso, lasso_oof_predictions) ** 0.5,
    "MAE": mean_absolute_error(y_lasso, lasso_oof_predictions),
    "spearman": float(spearmanr(y_lasso, lasso_oof_predictions)[0]),
}
lasso_fold_results = pd.DataFrame(lasso_fold_results)
display(pd.DataFrame([lasso_cv_metrics], index=["Lasso out-of-fold"]))
display(lasso_fold_results)

### Step 6 — Extract selected coefficients

Map encoded coefficients back to their original variables. A categorical variable is selected when at least one of its encoded levels has a nonzero coefficient. Family coverage is calculated from original variables.

In [ ]:
def encoded_feature_metadata(fitted_preprocessor, numerical_features=None, categorical_features=None):
    numerical_features = lasso_numerical_columns if numerical_features is None else numerical_features
    categorical_features = lasso_categorical_columns if categorical_features is None else categorical_features
    rows = []
    if numerical_features:
        rows.extend({"encoded_feature": variable, "original_variable": variable,
                     "category_level": np.nan}
                    for variable in numerical_features)
    if categorical_features:
        encoder = fitted_preprocessor.named_transformers_["categorical"].named_steps["encoder"]
        if hasattr(encoder, "get_feature_names_out"):
            encoded_names = list(encoder.get_feature_names_out(categorical_features))
        else:
            encoded_names = list(encoder.get_feature_names(categorical_features))
        original_names = []
        category_levels = []
        dropped_indices = (encoder.drop_idx_ if encoder.drop_idx_ is not None
                           else [None] * len(encoder.categories_))
        for variable, categories, dropped_index in zip(
            categorical_features, encoder.categories_, dropped_indices
        ):
            for index, category in enumerate(categories):
                if dropped_index is not None and index == dropped_index:
                    continue
                original_names.append(variable)
                category_levels.append(category)
        rows.extend({"encoded_feature": encoded, "original_variable": original,
                     "category_level": level}
                    for encoded, original, level in zip(
                        encoded_names, original_names, category_levels
                    ))
    return pd.DataFrame(rows)


lasso_coefficient_table = encoded_feature_metadata(
    lasso_net_pipeline.named_steps["preprocessor"]
)
lasso_coefficient_table["coefficient"] = lasso_net_model.coef_
lasso_coefficient_table["selected"] = (
    lasso_coefficient_table["coefficient"].abs() > LASSO_ZERO_TOLERANCE
)
lasso_selected_variables = lasso_coefficient_table.loc[
    lasso_coefficient_table["selected"], "original_variable"
].drop_duplicates().tolist()
lasso_selected_variable_table = pd.DataFrame({"Variable": lasso_selected_variables})
lasso_selected_variable_table["Feature_family"] = lasso_selected_variable_table["Variable"].map(
    feature_to_family
).fillna("geography")
selected_families = set(lasso_selected_variable_table["Feature_family"])
lasso_family_coverage = sum(family in selected_families for family in market_families)

display(lasso_selected_variable_table)
display(lasso_coefficient_table.loc[lasso_coefficient_table["selected"]])

### Step 7 — Calculate Lasso diagnostics

Calculate out-of-fold and in-sample predictive metrics. Adjusted R-squared uses the number of nonzero encoded coefficients as an effective parameter count. Classical inferential statistics are deferred to the post-selection OLS model.

In [ ]:
lasso_in_sample_predictions = lasso_net_pipeline.predict(X_lasso)
lasso_residuals = y_lasso - lasso_in_sample_predictions
lasso_n = len(y_lasso)
lasso_k = int(lasso_coefficient_table["selected"].sum())
lasso_r2 = r2_score(y_lasso, lasso_in_sample_predictions)
lasso_adjusted_r2 = (
    1 - (1 - lasso_r2) * (lasso_n - 1) / (lasso_n - lasso_k - 1)
    if lasso_n > lasso_k + 1 else np.nan
)

sign_lookup_series = feature_sign_lookup.set_index("Variable")["Expected_sign"]
lasso_sign_failures = []
for _, row in lasso_coefficient_table.loc[lasso_coefficient_table["selected"]].iterrows():
    expected = sign_lookup_series.get(row["original_variable"], "unconstrained")
    if ((expected == "positive" and row["coefficient"] <= 0) or
            (expected == "negative" and row["coefficient"] >= 0)):
        lasso_sign_failures.append(row["original_variable"])

lasso_in_sample_metrics = {
    "r_squared": lasso_r2,
    "adjusted_r_squared": lasso_adjusted_r2,
    "RMSE": mean_squared_error(y_lasso, lasso_in_sample_predictions) ** 0.5,
    "MAE": mean_absolute_error(y_lasso, lasso_in_sample_predictions),
    "spearman": float(spearmanr(y_lasso, lasso_in_sample_predictions)[0]),
    "sign_check": "FAIL" if lasso_sign_failures else "PASS",
    "sign_failures": ", ".join(sorted(set(lasso_sign_failures))),
    "n_observations": lasso_n,
    "n_original_variables": len(lasso_selected_variables),
    "n_parameters": lasso_k,
    "family_coverage": lasso_family_coverage,
    "families_included": ", ".join(sorted(selected_families)),
}
lasso_model_summary = pd.DataFrame([lasso_in_sample_metrics])
display(lasso_model_summary)

### Step 8 — Fit post-selection OLS diagnostics

Refit OLS using the original variables selected by Lasso. Numerical variables receive t-tests and categorical variables receive joint F-tests. Add the resulting variable-level p-value and the Lasso coefficient sign check to the coefficient export. These statistics are exploratory because selection and refitting use the same observations.

In [ ]:
post_selection_variables = []
post_selection_exclusion_rows = []
variable_strength = (
    lasso_coefficient_table.loc[lasso_coefficient_table["selected"]]
    .assign(abs_coefficient=lambda frame: frame["coefficient"].abs())
    .groupby("original_variable")["abs_coefficient"].sum()
    .sort_values(ascending=False)
)
for variable in variable_strength.index:
    proposed_variables = post_selection_variables + [variable]
    proposed_design, _ = build_design_matrix(modeling_data, proposed_variables)
    if (np.linalg.matrix_rank(proposed_design.to_numpy()) < proposed_design.shape[1] or
            proposed_design.shape[1] >= len(modeling_data)):
        post_selection_exclusion_rows.append({
            "Variable": variable,
            "Reason": "Creates rank deficiency or insufficient residual degrees of freedom",
        })
    else:
        post_selection_variables.append(variable)
post_selection_exclusions = pd.DataFrame(post_selection_exclusion_rows)
post_selected_families = {
    feature_to_family.get(variable, "geography")
    for variable in post_selection_variables
}
post_family_coverage = sum(
    family in post_selected_families for family in market_families
)

if not post_selection_variables:
    post_selection_tests = pd.DataFrame()
    post_selection_coefficients = pd.DataFrame()
    post_selection_diagnostics = pd.DataFrame([{
        "model_status": "INVALID", "exclusion_reason": "Lasso selected no variables."
    }])
    post_selection_model = None
else:
    post_design, post_blocks = build_design_matrix(modeling_data, post_selection_variables)
    post_selection_model = sm.OLS(y_lasso, post_design).fit()
    post_predictions = post_selection_model.predict(post_design)
    post_residuals = y_lasso - post_predictions
    post_tests_dict = variable_significance_tests(
        post_selection_model, post_blocks, post_selection_variables
    )
    post_selection_tests = pd.DataFrame([
        {"Variable": variable, **post_tests_dict[variable]}
        for variable in post_selection_variables
    ])
    post_max_vif, post_mean_vif = calculate_vif(post_design)
    post_sign_check, post_sign_failures = evaluate_signs(
        post_selection_model, post_selection_variables
    )
    post_n = int(post_selection_model.nobs)
    post_p = len(post_selection_model.params)
    post_aicc_denominator = post_n - post_p - 1
    post_aicc = (post_selection_model.aic + 2 * post_p * (post_p + 1) /
                 post_aicc_denominator if post_aicc_denominator > 0 else np.nan)
    post_selection_diagnostics = pd.DataFrame([{
        "model_status": "VALID", "exclusion_reason": "",
        "r_squared": post_selection_model.rsquared,
        "adjusted_r_squared": post_selection_model.rsquared_adj,
        "RMSE": np.mean(np.square(post_residuals)) ** 0.5,
        "MAE": np.mean(np.abs(post_residuals)),
        "spearman": float(spearmanr(y_lasso, post_predictions)[0]),
        "f_statistics": post_selection_model.fvalue,
        "f_p_value": post_selection_model.f_pvalue,
        "aic": post_selection_model.aic, "aicc": post_aicc,
        "bic": post_selection_model.bic, "max_vif": post_max_vif,
        "mean_vif": post_mean_vif,
        "condition_number": post_selection_model.condition_number,
        "max_cooks_distance": np.max(post_selection_model.get_influence().cooks_distance[0]),
        "sign_check": post_sign_check, "sign_failures": post_sign_failures,
        "n_observations": post_n,
        "n_original_variables": len(post_selection_variables),
        "n_parameters": post_p,
        "family_coverage": post_family_coverage,
        "families_included": ", ".join(sorted(post_selected_families)),
    }])
    post_selection_coefficients = pd.DataFrame({
        "Coefficient": post_selection_model.params.index,
        "Estimate": post_selection_model.params.values,
        "T-statistic": post_selection_model.tvalues.values,
        "P-value": post_selection_model.pvalues.values,
    })

lasso_coefficient_output = lasso_coefficient_table.loc[
    lasso_coefficient_table["selected"]
].copy()
lasso_coefficient_output["expected_sign"] = (
    lasso_coefficient_output["original_variable"].map(sign_lookup_series)
    .fillna("unconstrained")
)
lasso_coefficient_output["sign_check"] = np.select(
    [
        (lasso_coefficient_output["expected_sign"] == "positive") &
        (lasso_coefficient_output["coefficient"] > 0),
        (lasso_coefficient_output["expected_sign"] == "negative") &
        (lasso_coefficient_output["coefficient"] < 0),
        lasso_coefficient_output["expected_sign"].isin(["positive", "negative"]),
    ],
    ["PASS", "PASS", "FAIL"],
    default="NOT_APPLICABLE",
)
if post_selection_tests.empty:
    lasso_coefficient_output["p_value_test_type"] = "NOT_TESTED"
    lasso_coefficient_output["p_value"] = np.nan
else:
    variable_test_lookup = post_selection_tests.rename(columns={
        "Variable": "original_variable",
        "test_type": "p_value_test_type",
    })[["original_variable", "p_value_test_type", "p_value"]]
    lasso_coefficient_output = lasso_coefficient_output.merge(
        variable_test_lookup, on="original_variable", how="left"
    )
    lasso_coefficient_output["p_value_test_type"] = (
        lasso_coefficient_output["p_value_test_type"].fillna("NOT_TESTED")
    )

display(Markdown("#### Lasso selections excluded from the OLS refit"))
display(post_selection_exclusions)
display(Markdown("#### Post-selection variable tests"))
display(post_selection_tests)
display(Markdown("#### Post-selection OLS diagnostics"))
display(post_selection_diagnostics)

### Step 9 — Compare model results

Compare the previous best OLS model with Lasso out-of-fold performance, final Lasso in-sample performance, and post-selection OLS diagnostics. The evaluation basis is labeled so in-sample and out-of-fold results are not treated as equivalent.

In [ ]:
comparison_columns = [
    "Model", "Evaluation", "r_squared", "adjusted_r_squared", "RMSE", "MAE",
    "spearman", "f_statistics", "f_p_value", "aic", "aicc", "bic",
    "max_vif", "mean_vif", "condition_number", "max_cooks_distance",
    "sign_check", "family_coverage", "n_parameters",
]

comparison_rows = []
if "best_model" in globals():
    comparison_rows.append({
        "Model": "Best brute-force OLS", "Evaluation": "In-sample",
        **{column: best_model.get(column, np.nan) for column in comparison_columns[2:]},
    })
comparison_rows.append({
    "Model": "Lasso", "Evaluation": "Out-of-fold",
    **lasso_cv_metrics, "n_parameters": lasso_k,
    "family_coverage": lasso_family_coverage,
})
comparison_rows.append({
    "Model": "Lasso", "Evaluation": "In-sample",
    **lasso_in_sample_metrics,
})
if post_selection_model is not None:
    comparison_rows.append({
        "Model": "Lasso post-selection OLS", "Evaluation": "In-sample exploratory",
        **post_selection_diagnostics.iloc[0].to_dict(),
    })

lasso_model_comparison = pd.DataFrame(comparison_rows).reindex(columns=comparison_columns)
display(lasso_model_comparison)
display(Markdown("#### Selected original variables"))
display(lasso_selected_variable_table)
display(Markdown("#### Selected encoded coefficients"))
display(lasso_coefficient_table.loc[lasso_coefficient_table["selected"]])

### Step 10 — Record reproducibility and robustness information

Record the selected hyperparameters, fold design, seed, convergence warnings, and selected-feature counts. This table documents how the result was produced and highlights convergence issues.

In [ ]:
lasso_reproducibility = pd.DataFrame({
    "Setting": [
        "Selected alpha", "CV type", "Outer folds",
        "ZIP grouping used", "Random seed", "Alpha candidates",
        "Nonzero encoded coefficients", "Selected original variables",
        "Final-fit convergence warnings", "Outer-fold convergence warnings",
    ],
    "Value": [
        lasso_net_model.alpha_, lasso_cv_type,
        LASSO_N_SPLITS, zip_repeats, LASSO_RANDOM_SEED, len(LASSO_ALPHAS),
        lasso_k, len(lasso_selected_variables),
        len(lasso_convergence_warnings), len(lasso_oof_warnings),
    ],
})
display(lasso_reproducibility)
if lasso_convergence_warnings or lasso_oof_warnings:
    display(Markdown("**Review required:** One or more Lasso fits issued a convergence warning."))

### Step 11 — Refit after removing sign failures

As a contingency model, remove every original variable that failed the initial Lasso sign check and rerun `LassoCV` on all modeling records. This is a single removal-and-refit cycle, not an iterative selection procedure. The refit coefficient output repeats the initial coefficient format, including exploratory post-selection OLS p-values and updated sign checks.

In [ ]:
failed_sign_variables = sorted(set(lasso_sign_failures))
refit_lasso_numerical_columns = [
    variable for variable in lasso_numerical_columns
    if variable not in failed_sign_variables
]
refit_lasso_categorical_columns = [
    variable for variable in lasso_categorical_columns
    if variable not in failed_sign_variables
]
refit_lasso_features = refit_lasso_numerical_columns + refit_lasso_categorical_columns
if not refit_lasso_features:
    raise ValueError("No variables remain after removing sign-check failures.")

X_refit_lasso = modeling_data[refit_lasso_features].copy()
refit_inner_splits = make_inner_splits(X_refit_lasso, y_lasso, zip_groups)
refit_lasso_net_pipeline = Pipeline([
    ("preprocessor", make_lasso_preprocessor(
        refit_lasso_numerical_columns, refit_lasso_categorical_columns
    )),
    ("model", LassoCV(
        alphas=LASSO_ALPHAS, cv=refit_inner_splits,
        max_iter=100000, tol=1e-6, selection="cyclic", n_jobs=-1,
    )),
])
with warnings.catch_warnings(record=True) as refit_fit_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    refit_lasso_net_pipeline.fit(X_refit_lasso, y_lasso)
refit_lasso_net_model = refit_lasso_net_pipeline.named_steps["model"]

def refit_encoded_feature_metadata(fitted_preprocessor, numerical_features, categorical_features):
    rows = [
        {"encoded_feature": variable, "original_variable": variable,
         "category_level": np.nan}
        for variable in numerical_features
    ]
    if categorical_features:
        encoder = fitted_preprocessor.named_transformers_["categorical"].named_steps["encoder"]
        if hasattr(encoder, "get_feature_names_out"):
            encoded_names = list(encoder.get_feature_names_out(categorical_features))
        else:
            encoded_names = list(encoder.get_feature_names(categorical_features))
        dropped_indices = (encoder.drop_idx_ if encoder.drop_idx_ is not None
                           else [None] * len(encoder.categories_))
        metadata = []
        for variable, categories, dropped_index in zip(
            categorical_features, encoder.categories_, dropped_indices
        ):
            metadata.extend(
                (variable, category) for index, category in enumerate(categories)
                if dropped_index is None or index != dropped_index
            )
        rows.extend(
            {"encoded_feature": encoded, "original_variable": variable,
             "category_level": category}
            for encoded, (variable, category) in zip(encoded_names, metadata)
        )
    return pd.DataFrame(rows)


refit_lasso_coefficient_table = refit_encoded_feature_metadata(
    refit_lasso_net_pipeline.named_steps["preprocessor"],
    refit_lasso_numerical_columns, refit_lasso_categorical_columns,
)
refit_lasso_coefficient_table["coefficient"] = refit_lasso_net_model.coef_
refit_lasso_coefficient_table["selected"] = (
    refit_lasso_coefficient_table["coefficient"].abs() > LASSO_ZERO_TOLERANCE
)
refit_lasso_selected_variables = refit_lasso_coefficient_table.loc[
    refit_lasso_coefficient_table["selected"], "original_variable"
].drop_duplicates().tolist()

refit_post_selection_tests = pd.DataFrame()
if refit_lasso_selected_variables:
    refit_post_design, refit_post_blocks = build_design_matrix(
        modeling_data, refit_lasso_selected_variables
    )
    if np.linalg.matrix_rank(refit_post_design.to_numpy()) == refit_post_design.shape[1]:
        refit_post_model = sm.OLS(y_lasso, refit_post_design).fit()
        refit_test_dictionary = variable_significance_tests(
            refit_post_model, refit_post_blocks, refit_lasso_selected_variables
        )
        refit_post_selection_tests = pd.DataFrame([
            {"Variable": variable, **refit_test_dictionary[variable]}
            for variable in refit_lasso_selected_variables
        ])

refit_lasso_net_coefficients = refit_lasso_coefficient_table.loc[
    refit_lasso_coefficient_table["selected"]
].copy()
refit_lasso_net_coefficients["expected_sign"] = (
    refit_lasso_net_coefficients["original_variable"].map(sign_lookup_series)
    .fillna("unconstrained")
)
refit_lasso_net_coefficients["sign_check"] = np.select(
    [
        (refit_lasso_net_coefficients["expected_sign"] == "positive") &
        (refit_lasso_net_coefficients["coefficient"] > 0),
        (refit_lasso_net_coefficients["expected_sign"] == "negative") &
        (refit_lasso_net_coefficients["coefficient"] < 0),
        refit_lasso_net_coefficients["expected_sign"].isin(["positive", "negative"]),
    ],
    ["PASS", "PASS", "FAIL"], default="NOT_APPLICABLE",
)
if refit_post_selection_tests.empty:
    refit_lasso_net_coefficients["p_value_test_type"] = "NOT_TESTED"
    refit_lasso_net_coefficients["p_value"] = np.nan
else:
    refit_test_lookup = refit_post_selection_tests.set_index("Variable")
    refit_lasso_net_coefficients["p_value_test_type"] = (
        refit_lasso_net_coefficients["original_variable"].map(refit_test_lookup["test_type"])
        .fillna("NOT_TESTED")
    )
    refit_lasso_net_coefficients["p_value"] = (
        refit_lasso_net_coefficients["original_variable"].map(refit_test_lookup["p_value"])
    )

refit_predictions = refit_lasso_net_pipeline.predict(X_refit_lasso)
refit_n = len(y_lasso)
refit_k = int(refit_lasso_net_coefficients.shape[0])
refit_r_squared = r2_score(y_lasso, refit_predictions)
refit_adjusted_r_squared = (
    1 - (1 - refit_r_squared) * (refit_n - 1) / (refit_n - refit_k - 1)
    if refit_n > refit_k + 1 else np.nan
)
refit_selected_families = {
    feature_to_family.get(variable, "geography")
    for variable in refit_lasso_selected_variables
}
refit_sign_failures = sorted(set(
    refit_lasso_net_coefficients.loc[
        refit_lasso_net_coefficients["sign_check"] == "FAIL", "original_variable"
    ]
))
refit_lasso_net_model_summary = pd.DataFrame([{
    "r_squared": refit_r_squared,
    "adjusted_r_squared": refit_adjusted_r_squared,
    "RMSE": mean_squared_error(y_lasso, refit_predictions) ** 0.5,
    "MAE": mean_absolute_error(y_lasso, refit_predictions),
    "spearman": float(spearmanr(y_lasso, refit_predictions)[0]),
    "sign_check": "FAIL" if refit_sign_failures else "PASS",
    "sign_failures": ", ".join(refit_sign_failures),
    "n_observations": refit_n,
    "n_original_variables": len(refit_lasso_selected_variables),
    "n_parameters": refit_k,
    "family_coverage": sum(
        family in refit_selected_families for family in market_families
    ),
    "families_included": ", ".join(sorted(refit_selected_families)),
    "selected_alpha": refit_lasso_net_model.alpha_,
    "cv_RMSE": np.nan, "cv_MAE": np.nan,
    "cv_r_squared": np.nan, "cv_spearman": np.nan,
}])

# Plot standardized Lasso coefficients as directional feature importance.
import matplotlib.pyplot as plt

refit_feature_importance = refit_lasso_net_coefficients.copy()
refit_feature_importance["absolute_coefficient"] = (
    refit_feature_importance["coefficient"].abs()
)
refit_feature_importance = refit_feature_importance.sort_values(
    "absolute_coefficient", ascending=True
)
importance_colors = np.where(
    refit_feature_importance["coefficient"] >= 0, "#2E7D32", "#C44E00"
)
importance_figure, importance_axis = plt.subplots(figsize=(10, 6))
importance_axis.barh(
    refit_feature_importance["encoded_feature"],
    refit_feature_importance["coefficient"],
    color=importance_colors, alpha=0.9,
)
importance_axis.axvline(0, color="#171512", linewidth=1)
importance_axis.set(
    title="Refitted LassoCV feature importance",
    xlabel="Standardized coefficient (signed score-point contribution)",
    ylabel="Selected feature",
)
importance_axis.grid(axis="x", alpha=0.2)
importance_figure.tight_layout()
plots_dir = ROOT / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
importance_figure.savefig(
    plots_dir / "refitted_lasso_cv_feature_importance.png",
    dpi=300, bbox_inches="tight",
)
plt.show()

display(pd.DataFrame({"Removed for initial sign failure": failed_sign_variables}))
display(refit_lasso_net_model_summary)
display(refit_lasso_net_coefficients)

### Step 12 — Save Lasso results

Save Lasso outputs alongside, without overwriting, the existing brute-force model-search files. No files are written to `data` or `_raw`.

In [ ]:
lasso_output_dir = ROOT / "outputs" / "model_search"
lasso_output_dir.mkdir(parents=True, exist_ok=True)

lasso_compatibility_decisions.to_csv(
    lasso_output_dir / "lasso_compatibility_decisions.csv", index=False
)

lasso_model_summary.assign(
    selected_alpha=lasso_net_model.alpha_,
    cv_RMSE=lasso_cv_metrics["RMSE"],
    cv_MAE=lasso_cv_metrics["MAE"],
    cv_r_squared=lasso_cv_metrics["r_squared"],
    cv_spearman=lasso_cv_metrics["spearman"],
).to_csv(lasso_output_dir / "lasso_net_model_summary.csv", index=False)
lasso_coefficient_output.to_csv(
    lasso_output_dir / "lasso_net_coefficients.csv", index=False
)
lasso_selected_variable_table.to_csv(
    lasso_output_dir / "lasso_net_selected_variables.csv", index=False
)
post_selection_tests.to_csv(
    lasso_output_dir / "lasso_net_post_selection_tests.csv", index=False
)
lasso_model_comparison.to_csv(
    lasso_output_dir / "lasso_net_model_comparison.csv", index=False
)
refit_lasso_net_coefficients.to_csv(
    lasso_output_dir / "refit_lasso_net_coefficients.csv", index=False
)
refit_lasso_net_model_summary.to_csv(
    lasso_output_dir / "refit_lasso_net_model_summary.csv", index=False
)

print(f"Saved Lasso outputs to: {lasso_output_dir}")

## Market_Score_hat_latest

Use the baseline OLS and refitted LassoCV models to generate integer score estimates, enforce the 0–100 score range, name the champion output `Market_Score_hat_latest`, and save observed-versus-predicted plots in the `plots` folder.

In [ ]:
# Generate fitted Market Score estimates on the labeled modeling sample.
import matplotlib.pyplot as plt
import seaborn as sns

plots_dir = ROOT / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

market_score_hat = modeling_data[[GROUP_COLUMN, TARGET_COLUMN]].copy()
if "PERIOD_END" in modeling_data.columns:
    market_score_hat.insert(1, "PERIOD_END", modeling_data["PERIOD_END"])
market_score_hat["Market_Score_hat_baseline_OLS"] = (
    best_fitted_model.predict(best_design).to_numpy()
)
market_score_hat["Market_Score_hat_refit_LassoCV"] = (
    refit_lasso_net_pipeline.predict(modeling_data[refit_lasso_features])
)
# Round all model scores to integers and enforce the 0-100 score range.
model_prediction_columns = [
    "Market_Score_hat_baseline_OLS", "Market_Score_hat_refit_LassoCV"
]
for prediction_column in model_prediction_columns:
    below_floor = int((market_score_hat[prediction_column] < 0).sum())
    above_cap = int((market_score_hat[prediction_column] > 100).sum())
    market_score_hat[prediction_column] = (
        market_score_hat[prediction_column].round().clip(lower=0, upper=100).astype(int)
    )
    if below_floor or above_cap:
        print(
            f"{prediction_column}: score-range rule triggered; "
            f"{below_floor} prediction(s) floored at 0 and "
            f"{above_cap} prediction(s) capped at 100."
        )

# The sign-compliant refitted LassoCV is the latest-period champion score.
market_score_hat["Market_Score_hat_latest"] = (
    market_score_hat["Market_Score_hat_refit_LassoCV"]
)

prediction_specs = {
    "Baseline OLS": "Market_Score_hat_baseline_OLS",
    "Refitted LassoCV": "Market_Score_hat_refit_LassoCV",
}
prediction_plot_files = {
    "Baseline OLS": "baseline_OLS_Market_Score_hat_vs_Market_Score.png",
    "Refitted LassoCV": "refit_LassoCV_Market_Score_hat_vs_Market_Score.png",
}

prediction_metric_rows = []
figure, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
actual_score = market_score_hat[TARGET_COLUMN].astype(float)
score_min = min(actual_score.min(), market_score_hat[list(prediction_specs.values())].min().min())
score_max = max(actual_score.max(), market_score_hat[list(prediction_specs.values())].max().max())
plot_padding = max((score_max - score_min) * 0.05, 1)
plot_limits = (score_min - plot_padding, score_max + plot_padding)

for axis, (model_name, prediction_column) in zip(axes, prediction_specs.items()):
    predicted_score = market_score_hat[prediction_column]
    model_rmse = mean_squared_error(actual_score, predicted_score) ** 0.5
    model_mae = mean_absolute_error(actual_score, predicted_score)
    model_r_squared = r2_score(actual_score, predicted_score)
    model_spearman = float(spearmanr(actual_score, predicted_score)[0])
    prediction_metric_rows.append({
        "Model": model_name, "RMSE": model_rmse, "MAE": model_mae,
        "R_squared": model_r_squared, "Spearman": model_spearman,
    })

    sns.scatterplot(x=actual_score, y=predicted_score, alpha=0.7, s=45, ax=axis)
    axis.plot(plot_limits, plot_limits, color="#C44E52", linewidth=2, linestyle="--")
    axis.set(xlim=plot_limits, ylim=plot_limits, xlabel="Market_Score",
             ylabel="Market_Score_hat (integer, 0-100)", title=model_name)
    axis.text(
        0.04, 0.96,
        f"RMSE = {model_rmse:.2f}\nR² = {model_r_squared:.3f}\nSpearman = {model_spearman:.3f}",
        transform=axis.transAxes, va="top",
        bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
    )

    individual_figure, individual_axis = plt.subplots(figsize=(7, 6))
    sns.scatterplot(
        x=actual_score, y=predicted_score, alpha=0.7, s=45, ax=individual_axis
    )
    individual_axis.plot(
        plot_limits, plot_limits, color="#C44E52", linewidth=2, linestyle="--"
    )
    individual_axis.set(
        xlim=plot_limits, ylim=plot_limits, xlabel="Market_Score",
        ylabel="Market_Score_hat (integer, 0-100)",
        title=f"{model_name}: fitted vs. observed",
    )
    individual_axis.text(
        0.04, 0.96,
        f"RMSE = {model_rmse:.2f}\nR² = {model_r_squared:.3f}\nSpearman = {model_spearman:.3f}",
        transform=individual_axis.transAxes, va="top",
        bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
    )
    individual_figure.tight_layout()
    individual_figure.savefig(
        plots_dir / prediction_plot_files[model_name], dpi=300, bbox_inches="tight"
    )
    plt.close(individual_figure)

figure.suptitle("Observed vs. fitted Market Score", fontsize=15)
figure.tight_layout()
figure.savefig(
    plots_dir / "model_Market_Score_hat_comparison.png", dpi=300, bbox_inches="tight"
)
plt.show()

prediction_metrics = pd.DataFrame(prediction_metric_rows)
display(prediction_metrics)
display(market_score_hat.head())

In [ ]:
# Compare the latest-period champion score with its three strongest business drivers.
top_driver_variables = [
    "MEDIAN_DOM", "OFF_MARKET_IN_TWO_WEEKS", "SOLD_ABOVE_LIST"
]

latest_reference_rank_data = modeling_data[[GROUP_COLUMN] + top_driver_variables].copy()
latest_reference_rank_data["MarketScore_hat"] = market_score_hat[
    "Market_Score_hat_latest"
].to_numpy()

latest_all_rank_data = has_2025_data.loc[
    has_2025_data["has_2025"].eq(True)
    & has_2025_data["latest_period"].eq(True),
    [GROUP_COLUMN] + top_driver_variables + refit_lasso_features,
].copy()
latest_all_rank_data = latest_all_rank_data.loc[
    :, ~latest_all_rank_data.columns.duplicated()
]
latest_all_rank_data["MarketScore_hat"] = np.rint(np.clip(
    refit_lasso_net_pipeline.predict(latest_all_rank_data[refit_lasso_features]),
    0, 100,
)).astype(int)

def plot_latest_score_extremes(data, population_label, output_name):
    plot_data = data[[GROUP_COLUMN, "MarketScore_hat"] + top_driver_variables].copy()
    plot_data[GROUP_COLUMN] = plot_data[GROUP_COLUMN].astype(str).str.zfill(5)
    plot_data = plot_data.sort_values(["MarketScore_hat", GROUP_COLUMN])
    groups = [
        ("Bottom 10 ZIPs", plot_data.head(10).sort_values("MarketScore_hat", ascending=False)),
        ("Top 10 ZIPs", plot_data.tail(10).sort_values("MarketScore_hat")),
    ]
    colors = {
        "MarketScore_hat": "#1B5E20", "MEDIAN_DOM": "#C44E00",
        "OFF_MARKET_IN_TWO_WEEKS": "#E67E22", "SOLD_ABOVE_LIST": "#F5A623",
    }
    figure, axes = plt.subplots(2, 1, figsize=(15, 11), sharey=False)
    percentage_axes = []
    for axis, (panel_title, subset) in zip(axes, groups):
        x_positions = np.arange(len(subset))
        percentage_axis = axis.twinx()
        percentage_axes.append(percentage_axis)
        axis.plot(x_positions, subset["MarketScore_hat"], color=colors["MarketScore_hat"],
                  marker="D", linewidth=2.5, label="MarketScore_hat", zorder=4)
        axis.plot(x_positions, subset["MEDIAN_DOM"], color=colors["MEDIAN_DOM"],
                  marker="o", linewidth=1.8, linestyle=":", label="MEDIAN_DOM")
        for variable, marker in zip(
            ["OFF_MARKET_IN_TWO_WEEKS", "SOLD_ABOVE_LIST"], ["s", "^"]
        ):
            percentage_axis.plot(
                x_positions, subset[variable] * 100, color=colors[variable], marker=marker,
                linewidth=1.8, linestyle=":", label=variable,
            )
        axis.set_xticks(x_positions, subset[GROUP_COLUMN], rotation=45, ha="right")
        axis.set_ylim(bottom=0)
        percentage_axis.set_ylim(0, 100)
        percentage_axis.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(100))
        axis.grid(axis="y", alpha=0.25)
        axis.set_title(panel_title)
        axis.set_xlabel("ZIP Code")
        axis.set_ylabel("MarketScore_hat / Median DOM (days)")
        percentage_axis.set_ylabel("Share of homes")
    handles, labels = axes[0].get_legend_handles_labels()
    secondary_handles, secondary_labels = percentage_axes[0].get_legend_handles_labels()
    handles += secondary_handles
    labels += secondary_labels
    figure.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
    figure.suptitle(f"Latest-period score extremes and key market signals: {population_label}", fontsize=15)
    figure.tight_layout(rect=[0, 0.07, 1, 0.96])
    figure.savefig(plots_dir / output_name, dpi=300, bbox_inches="tight")
    plt.show()

plot_latest_score_extremes(
    latest_reference_rank_data, "selected reference ZIPs",
    "latest_selected_reference_top_bottom_score_drivers.png",
)
plot_latest_score_extremes(
    latest_all_rank_data, "all 524 SoCal ZIPs",
    "latest_all_zips_top_bottom_score_drivers.png",
)


## Market_Score_hat_all_zip

1. Now fit to all zips that has 2025 and has a market score. Use the baseline OLS and refitted LassoCV models to generate integer score estimates market_score_hat, enforce the 0–100 score range, name the champion output `Market_Score_hat_all_zip`, and save observed-versus-predicted plots in the `plots` folder.
2. Define PERIOD as the year-month of PERIOD_END 
3. select a zip that has most observation in the data above, plot the predicted score vs PERIOD, market score vs PERIOD, the topic three most significant varibale vs PERIOD in the same plot for earch of the baseline OLS and refitted LassoCV. Display and save the plots in the `plots` folder.
4. create the following two files:
    - scores_latest_select_zips_eval.csv: Zip Code, City, County, PERIOD, y_true, y_pred
    - scores_latest_all_zips.csv: Zip Code, City, County, PERIOD, MarketScore_hat

In [ ]:
# Prepare the scored history and reusable scoring helpers.
scored_2025_history = has_2025_data.loc[
    has_2025_data["has_2025"].eq(True) & has_2025_data[TARGET_COLUMN].notna()
].copy()
scored_2025_history["PERIOD_END"] = pd.to_datetime(
    scored_2025_history["PERIOD_END"], errors="raise"
)
scored_2025_history["PERIOD"] = (
    scored_2025_history["PERIOD_END"].dt.to_period("M").astype(str)
)

def build_ols_scoring_design(data):
    # Use training-sample imputations and the fitted model's exact columns.
    design_parts = []
    for variable in best_model_variables:
        if variable in categorical_columns:
            training_values = modeling_data[variable].astype("string")
            fill_value = training_values.mode(dropna=True).iloc[0]
            values = data[variable].astype("string").fillna(fill_value)
            design_parts.append(pd.get_dummies(
                values, prefix=variable, prefix_sep="=", dtype=float
            ))
        else:
            training_median = pd.to_numeric(
                modeling_data[variable], errors="coerce"
            ).median()
            values = pd.to_numeric(data[variable], errors="coerce").fillna(
                training_median
            )
            design_parts.append(values.rename(variable).to_frame())
    design = pd.concat(design_parts, axis=1).astype(float)
    design.insert(0, "const", 1.0)
    return design.reindex(
        columns=best_fitted_model.model.exog_names, fill_value=0.0
    )

def constrain_market_scores(raw_predictions, label):
    raw_predictions = pd.Series(raw_predictions)
    below_floor = int(raw_predictions.lt(0).sum())
    above_cap = int(raw_predictions.gt(100).sum())
    constrained = raw_predictions.round().clip(0, 100).astype(int)
    if below_floor or above_cap:
        print(
            f"{label}: score-range rule triggered; {below_floor} prediction(s) "
            f"floored at 0 and {above_cap} prediction(s) capped at 100."
        )
    return constrained.to_numpy()

print(
    f"Scoring {len(scored_2025_history):,} historical rows from "
    f"{scored_2025_history[GROUP_COLUMN].nunique():,} ZIP codes with 2025 data "
    "and an available reference score."
)

In [ ]:
# Apply the already fitted models to the scored history; do not retrain on repeated ZIP labels.
history_ols_design = build_ols_scoring_design(scored_2025_history)
scored_2025_history["Market_Score_hat_baseline_OLS"] = constrain_market_scores(
    best_fitted_model.predict(history_ols_design), "Baseline OLS history"
)
scored_2025_history["Market_Score_hat_refit_LassoCV"] = constrain_market_scores(
    refit_lasso_net_pipeline.predict(
        scored_2025_history[refit_lasso_features]
    ),
    "Refitted LassoCV history",
)
scored_2025_history["Market_Score_hat_all_zip"] = (
    scored_2025_history["Market_Score_hat_refit_LassoCV"]
)

history_prediction_specs = {
    "Baseline OLS": "Market_Score_hat_baseline_OLS",
    "Refitted LassoCV": "Market_Score_hat_refit_LassoCV",
}
history_metrics = []
figure, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
history_actual = scored_2025_history[TARGET_COLUMN].astype(float)
latest_history_mask = scored_2025_history["latest_period"].eq(True)
history_period_groups = [
    (~latest_history_mask, "Earlier period", "#ebeb2a"),
    (latest_history_mask, "Latest period", "#2a5aeb"),
]
for axis, (model_name, prediction_column) in zip(
    axes, history_prediction_specs.items()
):
    history_predicted = scored_2025_history[prediction_column].astype(float)
    metrics = {
        "Model": model_name,
        "Rows": len(scored_2025_history),
        "RMSE": mean_squared_error(history_actual, history_predicted) ** 0.5,
        "MAE": mean_absolute_error(history_actual, history_predicted),
        "R_squared": r2_score(history_actual, history_predicted),
        "Spearman": float(spearmanr(history_actual, history_predicted)[0]),
    }
    history_metrics.append(metrics)
    for period_mask, period_label, period_color in history_period_groups:
        sns.scatterplot(
            x=history_actual.loc[period_mask],
            y=history_predicted.loc[period_mask],
            color=period_color, label=period_label, alpha=0.55, s=34, ax=axis,
        )
    axis.plot([0, 100], [0, 100], color="#C44E52", linestyle="--")
    axis.set(
        xlim=(0, 100), ylim=(0, 100), xlabel="Market_Score",
        ylabel="Predicted score (integer, 0-100)", title=model_name,
    )
    axis.text(
        0.04, 0.96,
        f"RMSE = {metrics['RMSE']:.2f}\nR² = {metrics['R_squared']:.3f}",
        transform=axis.transAxes, va="top",
        bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
    )
    individual_figure, individual_axis = plt.subplots(figsize=(7, 6))
    for period_mask, period_label, period_color in history_period_groups:
        sns.scatterplot(
            x=history_actual.loc[period_mask],
            y=history_predicted.loc[period_mask],
            color=period_color, label=period_label, alpha=0.55, s=34,
            ax=individual_axis,
        )
    individual_axis.plot(
        [0, 100], [0, 100], color="#C44E52", linestyle="--"
    )
    individual_axis.set(
        xlim=(0, 100), ylim=(0, 100), xlabel="Market_Score",
        ylabel="Predicted score (integer, 0-100)",
        title=f"{model_name}: all scored 2025-eligible ZIP history",
    )
    individual_figure.tight_layout()
    individual_figure.savefig(
        plots_dir / f"{model_name.replace(' ', '_')}_all_zip_vs_Market_Score.png",
        dpi=300, bbox_inches="tight",
    )
    plt.close(individual_figure)
figure.suptitle("Observed vs. predicted score: all scored 2025-eligible ZIP history")
figure.tight_layout()
figure.savefig(
    plots_dir / "Market_Score_hat_all_zip_comparison.png",
    dpi=300, bbox_inches="tight",
)
plt.show()
history_prediction_metrics = pd.DataFrame(history_metrics)
display(history_prediction_metrics)

### A Note

The negative historical R² should not be interpreted as evidence of poor historical predictive accuracy. Historical Market Scores are unavailable, so each ZIP’s latest-2025 reference score has been mapped to all of its earlier observations solely as a comparison benchmark. This creates a time-invariant target, while the model produces time-varying predictions from each period’s market conditions. Consequently, genuine changes in days on market, sales activity, and market competitiveness appear as prediction errors against the repeated 2025 score. The historical comparison is therefore a behavioral diagnostic rather than a valid accuracy test. Historical predictions should instead be assessed for stability, economically sensible direction, and gradual movement over time. Model accuracy against the reference score should be evaluated using latest-period observations only.

In [ ]:
scored_2025_history

In [ ]:
# Select the most-observed ZIP and the three lowest-p-value numerical drivers per model.
zip_observation_counts = (
    scored_2025_history.groupby(GROUP_COLUMN).size().rename("Observations")
    .reset_index().sort_values(
        ["Observations", GROUP_COLUMN], ascending=[False, True]
    )
)
trend_zip = zip_observation_counts.iloc[0][GROUP_COLUMN]

baseline_significance = pd.DataFrame({
    "Variable": best_model_variables,
    "p_value": [
        best_model[f"p_value_{position}"]
        for position in range(1, len(best_model_variables) + 1)
    ],
})
baseline_top_variables = (
    baseline_significance.loc[
        baseline_significance["Variable"].isin(numerical_columns)
    ].sort_values("p_value").head(3)["Variable"].tolist()
)
refit_significance = (
    refit_lasso_net_coefficients.loc[
        refit_lasso_net_coefficients["original_variable"].isin(numerical_columns)
    ].groupby("original_variable", as_index=False)["p_value"].min()
    .sort_values("p_value")
)
refit_top_variables = refit_significance.head(3)["original_variable"].tolist()

if len(baseline_top_variables) < 3 or len(refit_top_variables) < 3:
    raise ValueError("Each model must have at least three numerical variables for the trend plot.")

display(zip_observation_counts.head())
display(pd.DataFrame({
    "Model": ["Baseline OLS", "Refitted LassoCV"],
    "Trend ZIP": [trend_zip, trend_zip],
    "Top three variables": [
        ", ".join(baseline_top_variables), ", ".join(refit_top_variables)
    ],
}))

In [ ]:
# Plot reference and predicted scores with standardized driver paths on a secondary axis.
def plot_zip_score_and_drivers(data, zip_code, prediction_column, variables, model_name):
    plot_data = data.loc[data[GROUP_COLUMN].eq(zip_code)].copy()
    plot_data = plot_data.sort_values("PERIOD_END")
    plot_data["PERIOD_DATE"] = plot_data["PERIOD_END"].dt.to_period("M").dt.to_timestamp()

    figure, score_axis = plt.subplots(figsize=(14, 7))
    score_axis.plot(
        plot_data["PERIOD_DATE"], plot_data[TARGET_COLUMN],
        color="#1f77b4", linewidth=2.4, label="Market Score",
    )
    score_axis.plot(
        plot_data["PERIOD_DATE"], plot_data[prediction_column],
        color="#d62728", linewidth=2.0, label="Predicted score",
    )
    score_axis.set(
        xlabel="PERIOD", ylabel="Score (0-100)", ylim=(0, 100),
        title=f"{model_name}: ZIP {zip_code} score and top drivers over time",
    )

    driver_axis = score_axis.twinx()
    for variable in variables:
        values = pd.to_numeric(plot_data[variable], errors="coerce")
        standard_deviation = values.std(ddof=0)
        standardized = (
            (values - values.mean()) / standard_deviation
            if pd.notna(standard_deviation) and standard_deviation > 0
            else pd.Series(0.0, index=values.index)
        )
        driver_axis.plot(
            plot_data["PERIOD_DATE"], standardized,
            linewidth=1.3, alpha=0.8, linestyle="--", label=variable,
        )
    driver_axis.set_ylabel("Driver value (within-ZIP z-score)")
    score_axis.grid(alpha=0.2)
    score_lines, score_labels = score_axis.get_legend_handles_labels()
    driver_lines, driver_labels = driver_axis.get_legend_handles_labels()
    score_axis.legend(
        score_lines + driver_lines, score_labels + driver_labels,
        loc="best", frameon=True,
    )
    figure.autofmt_xdate()
    figure.tight_layout()
    output_file = plots_dir / (
        f"ZIP_{zip_code}_{model_name.replace(' ', '_')}_score_driver_trend.png"
    )
    figure.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.show()
    return output_file

baseline_trend_plot = plot_zip_score_and_drivers(
    scored_2025_history, trend_zip, "Market_Score_hat_baseline_OLS",
    baseline_top_variables, "Baseline OLS",
)
refit_lasso_trend_plot = plot_zip_score_and_drivers(
    scored_2025_history, trend_zip, "Market_Score_hat_refit_LassoCV",
    refit_top_variables, "Refitted LassoCV",
)
print(f"Saved trend plots to: {baseline_trend_plot} and {refit_lasso_trend_plot}")

In [ ]:
# Build a stable ZIP-to-city lookup for the required output columns.
raw_socal_identifiers = pd.read_csv(
    ROOT / "data" / "SoCal.csv", usecols=["Zip Code", "City"]
)
city_lookup = (
    raw_socal_identifiers.dropna(subset=["City"])
    .groupby("Zip Code")["City"]
    .agg(lambda values: values.mode().iloc[0])
)

latest_eval_data = modeling_data.copy()
latest_eval_data["PERIOD_END"] = pd.to_datetime(
    latest_eval_data["PERIOD_END"], errors="raise"
)
latest_eval_data["PERIOD"] = (
    latest_eval_data["PERIOD_END"].dt.to_period("M").astype(str)
)
latest_eval_data["City"] = latest_eval_data[GROUP_COLUMN].map(city_lookup)
latest_eval_data["y_pred"] = constrain_market_scores(
    refit_lasso_net_pipeline.predict(latest_eval_data[refit_lasso_features]),
    "Latest selected-ZIP refitted LassoCV",
)
scores_latest_select_zips_eval = latest_eval_data.rename(
    columns={TARGET_COLUMN: "y_true"}
)[[GROUP_COLUMN, "City", "County", "PERIOD", "y_true", "y_pred"]]
scores_latest_select_zips_eval["y_true"] = (
    scores_latest_select_zips_eval["y_true"].round().astype(int)
)

latest_all_zip_data = has_2025_data.loc[
    has_2025_data["has_2025"].eq(True)
    & has_2025_data["latest_period"].eq(True)
].copy()
latest_all_zip_data["PERIOD_END"] = pd.to_datetime(
    latest_all_zip_data["PERIOD_END"], errors="raise"
)
latest_all_zip_data["PERIOD"] = (
    latest_all_zip_data["PERIOD_END"].dt.to_period("M").astype(str)
)
latest_all_zip_data["City"] = latest_all_zip_data[GROUP_COLUMN].map(city_lookup)
latest_all_zip_data["MarketScore_hat"] = constrain_market_scores(
    refit_lasso_net_pipeline.predict(
        latest_all_zip_data[refit_lasso_features]
    ),
    "Latest all-ZIP refitted LassoCV",
)
latest_all_zip_data["Market_Score_hat_all_zip"] = (
    latest_all_zip_data["MarketScore_hat"]
)
scores_latest_all_zips = latest_all_zip_data[[
    GROUP_COLUMN, "City", "County", "PERIOD", "MarketScore_hat"
]].copy()

# Enriched copies add the champion model's selected explanatory variables.
selected_model_explanatory_variables = list(dict.fromkeys(
    refit_lasso_selected_variables
))
selected_eval_base_columns = [
    GROUP_COLUMN, "City", "County", "PERIOD", "y_true", "y_pred"
]
all_zip_base_columns = [
    GROUP_COLUMN, "City", "County", "PERIOD", "MarketScore_hat"
]
selected_eval_extra_columns = [
    variable for variable in selected_model_explanatory_variables
    if variable not in selected_eval_base_columns
]
all_zip_extra_columns = [
    variable for variable in selected_model_explanatory_variables
    if variable not in all_zip_base_columns
]
scores_latest_select_zips_eval_with_var = latest_eval_data.rename(
    columns={TARGET_COLUMN: "y_true"}
)[selected_eval_base_columns + selected_eval_extra_columns].copy()
scores_latest_select_zips_eval_with_var["y_true"] = (
    scores_latest_select_zips_eval_with_var["y_true"].round().astype(int)
)
scores_latest_all_zips_with_var = latest_all_zip_data[
    all_zip_base_columns + all_zip_extra_columns
].copy()

In [ ]:
# Validate and export the two required latest-period score files.
if scores_latest_select_zips_eval[GROUP_COLUMN].duplicated().any():
    raise ValueError("The selected-ZIP evaluation output contains duplicate ZIP codes.")
if scores_latest_all_zips[GROUP_COLUMN].duplicated().any():
    raise ValueError("The all-ZIP latest output contains duplicate ZIP codes.")
if scores_latest_select_zips_eval["City"].isna().any():
    raise ValueError("City is missing for one or more selected-ZIP output rows.")
if scores_latest_all_zips["City"].isna().any():
    raise ValueError("City is missing for one or more all-ZIP output rows.")
if not scores_latest_select_zips_eval["y_pred"].between(0, 100).all():
    raise ValueError("Selected-ZIP predictions are outside 0-100.")
if not scores_latest_all_zips["MarketScore_hat"].between(0, 100).all():
    raise ValueError("All-ZIP predictions are outside 0-100.")

score_output_dir = ROOT / "outputs"
score_output_dir.mkdir(parents=True, exist_ok=True)
selected_zip_output_path = score_output_dir / "scores_latest_select_zips_eval.csv"
all_zip_output_path = score_output_dir / "scores_latest_all_zips.csv"
selected_zip_with_var_output_path = (
    score_output_dir / "scores_latest_select_zips_eval_with_var.csv"
)
all_zip_with_var_output_path = (
    score_output_dir / "scores_latest_all_zips_with_var.csv"
)
scores_latest_select_zips_eval.to_csv(selected_zip_output_path, index=False)
scores_latest_all_zips.to_csv(all_zip_output_path, index=False)
scores_latest_select_zips_eval_with_var.to_csv(
    selected_zip_with_var_output_path, index=False
)
scores_latest_all_zips_with_var.to_csv(
    all_zip_with_var_output_path, index=False
)

print(f"Saved {len(scores_latest_select_zips_eval):,} rows to {selected_zip_output_path}")
print(f"Saved {len(scores_latest_all_zips):,} rows to {all_zip_output_path}")
print(
    f"Saved {len(scores_latest_select_zips_eval_with_var):,} rows to "
    f"{selected_zip_with_var_output_path}"
)
print(
    f"Saved {len(scores_latest_all_zips_with_var):,} rows to "
    f"{all_zip_with_var_output_path}"
)
display(scores_latest_select_zips_eval.head())
display(scores_latest_all_zips.head())

## Latest-period market-signal sanity check

Compare the champion score with its three lowest-p-value numerical explanatory variables. The first figure aligns all series by ZIP (ordered by predicted score); the second directly tests whether each signal moves with the score in its expected economic direction.

In [ ]:
# Display the score and top three significant variables across the latest ZIPs.
top_significant_variables = (
    refit_lasso_net_coefficients.loc[
        refit_lasso_net_coefficients["original_variable"].isin(numerical_columns)
    ]
    .groupby("original_variable", as_index=False)["p_value"].min()
    .sort_values("p_value")
    .head(3)["original_variable"].tolist()
)
if len(top_significant_variables) != 3:
    raise ValueError("Three numerical explanatory variables are required.")

zip_signal_plot_data = (
    scores_latest_all_zips_with_var
    .sort_values(["MarketScore_hat", GROUP_COLUMN])
    .reset_index(drop=True)
)
zip_signal_plot_data["ZIP_order"] = np.arange(len(zip_signal_plot_data))
panel_variables = ["MarketScore_hat"] + top_significant_variables
figure, axes = plt.subplots(
    len(panel_variables), 1, figsize=(16, 12), sharex=True
)
panel_colors = ["#2E8B57", "#4C92C3", "#DD8452", "#8172B2"]
for axis, variable, color in zip(axes, panel_variables, panel_colors):
    axis.scatter(
        zip_signal_plot_data["ZIP_order"],
        zip_signal_plot_data[variable],
        color=color, alpha=0.7, s=22,
    )
    axis.set_ylabel(variable)
    axis.grid(axis="y", alpha=0.2)
tick_step = max(len(zip_signal_plot_data) // 12, 1)
tick_positions = np.arange(0, len(zip_signal_plot_data), tick_step)
axes[-1].set_xticks(tick_positions)
axes[-1].set_xticklabels(
    zip_signal_plot_data.loc[tick_positions, GROUP_COLUMN].astype(str),
    rotation=45, ha="right",
)
axes[-1].set_xlabel("ZIP code (ordered by MarketScore_hat)")
figure.suptitle(
    "Latest-period score and top explanatory variables across ZIP codes",
    fontsize=15,
)
figure.tight_layout()
zip_signal_plot_path = plots_dir / "latest_all_zips_score_and_signals_by_zip.png"
figure.savefig(zip_signal_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved ZIP-aligned sanity-check plot to: {zip_signal_plot_path}")

In [ ]:
# Directly check monotonic alignment between each key signal and the champion score.
signal_check_rows = []
figure, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)
for axis, variable in zip(axes, top_significant_variables):
    valid = scores_latest_all_zips_with_var[[
        variable, "MarketScore_hat"
    ]].dropna()
    correlation = float(spearmanr(
        valid[variable], valid["MarketScore_hat"]
    )[0])
    expected_direction = sign_lookup_series.get(variable, "unconstrained")
    direction_pass = (
        correlation > 0 if expected_direction == "positive"
        else correlation < 0 if expected_direction == "negative"
        else True
    )
    signal_check_rows.append({
        "Variable": variable,
        "Expected direction": expected_direction,
        "Spearman correlation": correlation,
        "Direction check": "PASS" if direction_pass else "FAIL",
        "Rows": len(valid),
    })
    sns.regplot(
        data=valid, x=variable, y="MarketScore_hat", ax=axis,
        scatter_kws={"alpha": 0.55, "s": 28, "color": "#4C92C3"},
        line_kws={"color": "#C44E52", "linewidth": 2},
    )
    axis.set_title(
        f"{variable}\nSpearman = {correlation:.3f}; "
        f"direction {'PASS' if direction_pass else 'FAIL'}"
    )
    axis.set_ylabel("MarketScore_hat" if axis is axes[0] else "")
    axis.set_ylim(0, 105)
    axis.grid(alpha=0.2)
figure.suptitle("Latest-period champion score alignment with key market signals")
figure.tight_layout()
signal_alignment_plot_path = (
    plots_dir / "latest_all_zips_market_signal_alignment.png"
)
figure.savefig(signal_alignment_plot_path, dpi=300, bbox_inches="tight")
plt.show()
signal_alignment_summary = pd.DataFrame(signal_check_rows)
display(signal_alignment_summary)
print(f"Saved direct signal-alignment plot to: {signal_alignment_plot_path}")

In [ ]:
# Period-average trend plots for scores and the three strongest market signals.
from matplotlib.ticker import PercentFormatter

percentage_signal_variables = [
    "OFF_MARKET_IN_TWO_WEEKS", "SOLD_ABOVE_LIST"
]
primary_signal_variable = "MEDIAN_DOM"
expected_top_variables = {primary_signal_variable, *percentage_signal_variables}
if set(top_significant_variables) != expected_top_variables:
    raise ValueError(
        "The three strongest numerical variables changed; review the axis assignment."
    )

def plot_period_average_signals(data, score_columns, title, file_name):
    required_columns = [
        "PERIOD", primary_signal_variable,
        *percentage_signal_variables, *score_columns,
    ]
    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(f"Missing columns for period plot: {missing_columns}")

    period_data = data[required_columns].copy()
    period_data["PERIOD_DATE"] = pd.to_datetime(
        period_data["PERIOD"].astype(str) + "-01", errors="raise"
    )
    mean_columns = score_columns + [
        primary_signal_variable, *percentage_signal_variables
    ]
    period_summary = (
        period_data.groupby(["PERIOD_DATE", "PERIOD"], as_index=False)
        .agg(
            **{column: (column, "mean") for column in mean_columns},
            ZIP_count=("PERIOD", "size"),
        )
        .sort_values("PERIOD_DATE")
    )

    figure, primary_axis = plt.subplots(figsize=(13, 7))
    # Scores use green shades; explanatory variables use orange shades.
    score_colors = ["#1B5E20", "#43A047", "#81C784"]
    for score_column, color in zip(score_columns, score_colors):
        primary_axis.plot(
            period_summary["PERIOD_DATE"], period_summary[score_column],
            color=color, linewidth=2.4, marker="o", label=score_column,
        )
    primary_axis.plot(
        period_summary["PERIOD_DATE"],
        period_summary[primary_signal_variable],
        color="#C44E00", linewidth=2, linestyle=":", marker="s",
        label=f"Average {primary_signal_variable}",
    )
    primary_axis.set_ylabel("Average score / median days on market")
    primary_axis.set_xlabel("PERIOD")
    primary_axis.grid(alpha=0.2)

    percentage_axis = primary_axis.twinx()
    percentage_colors = ["#E67E22", "#F5A623"]
    percentage_markers = ["^", "D"]
    for variable, color, marker in zip(
        percentage_signal_variables, percentage_colors, percentage_markers
    ):
        percentage_axis.plot(
            period_summary["PERIOD_DATE"], period_summary[variable],
            color=color, linewidth=2, linestyle=":", marker=marker,
            label=f"Average {variable}",
        )
    percentage_axis.set_ylabel("Average percentage market signal")
    percentage_axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))

    primary_lines, primary_labels = primary_axis.get_legend_handles_labels()
    percentage_lines, percentage_labels = (
        percentage_axis.get_legend_handles_labels()
    )
    primary_axis.legend(
        primary_lines + percentage_lines, primary_labels + percentage_labels,
        loc="best", frameon=True,
    )
    primary_axis.set_title(title)
    primary_axis.set_xticks(period_summary["PERIOD_DATE"])
    primary_axis.set_xticklabels(
        period_summary["PERIOD"], rotation=45, ha="right"
    )
    figure.tight_layout()
    output_path = plots_dir / file_name
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return period_summary, output_path

selected_zip_period_summary, selected_zip_period_plot_path = (
    plot_period_average_signals(
        scores_latest_select_zips_eval_with_var,
        ["y_true", "y_pred"],
        "Selected scored ZIPs: average score and key signals by latest period",
        "selected_zips_average_score_and_signals_by_period.png",
    )
)
all_zip_period_summary, all_zip_period_plot_path = plot_period_average_signals(
    scores_latest_all_zips_with_var,
    ["MarketScore_hat"],
    "All ZIPs: average champion score and key signals by latest period",
    "all_zips_average_score_and_signals_by_period.png",
)

display(selected_zip_period_summary)
display(all_zip_period_summary)
print(
    f"Saved period-average plots to: {selected_zip_period_plot_path} and "
    f"{all_zip_period_plot_path}"
)

# Full-period Scoring

Using the same/optimized modeling approach (same preprocessing + fitted model), apply it to all rows in the SoCal dataset (all available time periods) to generate:
 - Market Score_hat for every existing (Zip Code, PERIOD) row.
 - A time-series plot of MarketScore_hat over time for ZIP Code (Visualization of at least one ZIP)

In [ ]:
# Apply the fixed champion pipeline to every existing SoCal ZIP-period row.
missing_full_period_features = sorted(
    set(refit_lasso_features) - set(full_period_data.columns)
)
if missing_full_period_features:
    raise ValueError(
        f"Full-period data is missing model features: {missing_full_period_features}"
    )

full_period_score_data = full_period_data.copy()
full_period_score_data["PERIOD_END"] = pd.to_datetime(
    full_period_score_data["PERIOD_END"], errors="raise"
)
full_period_score_data["PERIOD"] = (
    full_period_score_data["PERIOD_END"].dt.to_period("M").astype(str)
)
full_period_score_data["City"] = (
    full_period_score_data[GROUP_COLUMN].map(city_lookup)
)

full_period_raw_predictions = refit_lasso_net_pipeline.predict(
    full_period_score_data[refit_lasso_features]
)
full_period_score_data["MarketScore_hat"] = constrain_market_scores(
    full_period_raw_predictions, "Full-period refitted LassoCV"
)
full_period_clipping_summary = pd.DataFrame([{
    "Rows scored": len(full_period_score_data),
    "ZIP codes": full_period_score_data[GROUP_COLUMN].nunique(),
    "Periods": full_period_score_data["PERIOD"].nunique(),
    "Predictions floored at 0": int((full_period_raw_predictions < 0).sum()),
    "Predictions capped at 100": int((full_period_raw_predictions > 100).sum()),
}])
display(full_period_clipping_summary)

In [ ]:
# Validate the required one-row-per-ZIP-period output and export it.
full_period_key_columns = [GROUP_COLUMN, "PERIOD"]
if full_period_score_data.duplicated(full_period_key_columns).any():
    duplicate_keys = full_period_score_data.loc[
        full_period_score_data.duplicated(full_period_key_columns, keep=False),
        full_period_key_columns,
    ].sort_values(full_period_key_columns)
    raise ValueError(
        "Full-period output contains duplicate (Zip Code, PERIOD) keys.\n"
        + duplicate_keys.head(20).to_string(index=False)
    )
if full_period_score_data["City"].isna().any():
    raise ValueError("City is missing for one or more full-period rows.")
if not full_period_score_data["MarketScore_hat"].between(0, 100).all():
    raise ValueError("Full-period predictions are outside 0-100.")
if full_period_score_data["MarketScore_hat"].dtype.kind not in "iu":
    raise TypeError("Full-period MarketScore_hat must contain integers.")

scores_all_periods = (
    full_period_score_data[[
        GROUP_COLUMN, "City", "County", "PERIOD", "MarketScore_hat"
    ]]
    .sort_values([GROUP_COLUMN, "PERIOD"])
    .reset_index(drop=True)
)
scores_all_periods_output_path = score_output_dir / "scores_all_periods.csv"
scores_all_periods.to_csv(scores_all_periods_output_path, index=False)
print(
    f"Saved {len(scores_all_periods):,} rows to "
    f"{scores_all_periods_output_path}"
)

In [ ]:
# Summarize coverage and the constrained full-period score distribution.
full_period_output_summary = pd.DataFrame({
    "Measure": [
        "Rows", "Unique ZIP codes", "Unique periods",
        "Earliest period", "Latest period", "Minimum score",
        "Median score", "Maximum score",
    ],
    "Value": [
        len(scores_all_periods),
        scores_all_periods[GROUP_COLUMN].nunique(),
        scores_all_periods["PERIOD"].nunique(),
        scores_all_periods["PERIOD"].min(),
        scores_all_periods["PERIOD"].max(),
        scores_all_periods["MarketScore_hat"].min(),
        scores_all_periods["MarketScore_hat"].median(),
        scores_all_periods["MarketScore_hat"].max(),
    ],
})
display(full_period_output_summary)
display(scores_all_periods.head())

In [ ]:
# Plot the champion score over time for the ZIP with the most observations.
full_period_zip_counts = (
    full_period_score_data.groupby(GROUP_COLUMN).size().rename("Observations")
    .reset_index().sort_values(
        ["Observations", GROUP_COLUMN], ascending=[False, True]
    )
)
full_period_example_zip = full_period_zip_counts.iloc[0][GROUP_COLUMN]
full_period_example = (
    full_period_score_data.loc[
        full_period_score_data[GROUP_COLUMN].eq(full_period_example_zip)
    ]
    .sort_values("PERIOD_END")
    .copy()
)
example_city = full_period_example["City"].dropna().iloc[0]

full_period_top_variables = list(top_significant_variables)
if len(full_period_top_variables) != 3:
    raise ValueError("Three explanatory variables are required for the trend plot.")

figure, score_axis = plt.subplots(figsize=(13, 6))
score_axis.plot(
    full_period_example["PERIOD_END"],
    full_period_example["MarketScore_hat"],
    color="#1B5E20", linewidth=2.4, marker="o", markersize=5,
    label="MarketScore_hat",
)
score_axis.set(
    xlabel="PERIOD", ylabel="MarketScore_hat (0-100)", ylim=(0, 100),
    title=(
        f"Full-period Market Score: ZIP {full_period_example_zip} "
        f"({example_city})"
    ),
)
score_axis.grid(alpha=0.2)

# Standardize the drivers within this ZIP so different units share one axis.
driver_axis = score_axis.twinx()
driver_colors = ["#C44E00", "#E67E22", "#F5A623"]
driver_markers = ["s", "^", "D"]
for variable, color, marker in zip(
    full_period_top_variables, driver_colors, driver_markers
):
    values = pd.to_numeric(full_period_example[variable], errors="coerce")
    standard_deviation = values.std(ddof=0)
    standardized_values = (
        (values - values.mean()) / standard_deviation
        if pd.notna(standard_deviation) and standard_deviation > 0
        else pd.Series(0.0, index=values.index)
    )
    driver_axis.plot(
        full_period_example["PERIOD_END"], standardized_values,
        color=color, linewidth=1.8, linestyle=":", marker=marker,
        markersize=4, label=variable,
    )
driver_axis.set_ylabel("Explanatory variable (within-ZIP z-score)")

score_lines, score_labels = score_axis.get_legend_handles_labels()
driver_lines, driver_labels = driver_axis.get_legend_handles_labels()
score_axis.legend(
    score_lines + driver_lines, score_labels + driver_labels,
    loc="best", frameon=True,
)
score_axis.text(
    0.01, 0.02, "Existing observations only; missing periods are not imputed.",
    transform=score_axis.transAxes, color="#5f584f",
)
figure.autofmt_xdate()
figure.tight_layout()
full_period_plot_path = plots_dir / (
    f"full_period_MarketScore_hat_ZIP_{full_period_example_zip}.png"
)
figure.savefig(full_period_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(
    f"Saved {len(full_period_example)}-observation ZIP trend to: "
    f"{full_period_plot_path}"
)
display(full_period_zip_counts.head())

# Full-period Analysis

## Historical stability

Create measures and displays to quantify stability. include plots or tables for the following
- median annual score change
- percentage of ZIPs with large jumps
- maximum observation-to-observation changes.

In [ ]:
# Quantify full-period score stability using existing, irregular observations only.
LARGE_JUMP_THRESHOLD = 50
stability_data = scores_all_periods.copy()
stability_data["PERIOD_DATE"] = pd.to_datetime(
    stability_data["PERIOD"], format="%Y-%m", errors="raise"
)
stability_data = stability_data.sort_values([GROUP_COLUMN, "PERIOD_DATE"])
stability_data["Previous_period"] = stability_data.groupby(GROUP_COLUMN)["PERIOD"].shift()
stability_data["Previous_score"] = stability_data.groupby(GROUP_COLUMN)["MarketScore_hat"].shift()
stability_data["Observation_change"] = (
    stability_data["MarketScore_hat"] - stability_data["Previous_score"]
)
stability_data["Absolute_observation_change"] = stability_data["Observation_change"].abs()
stability_data["Days_since_previous"] = stability_data.groupby(GROUP_COLUMN)["PERIOD_DATE"].diff().dt.days

# Use each ZIP's final observation in each year and retain consecutive-year pairs only.
stability_data["YEAR"] = stability_data["PERIOD_DATE"].dt.year
annual_snapshots = (
    stability_data.sort_values([GROUP_COLUMN, "YEAR", "PERIOD_DATE"])
    .groupby([GROUP_COLUMN, "YEAR"], as_index=False).tail(1)
    .sort_values([GROUP_COLUMN, "YEAR"])
)
annual_snapshots["Previous_year"] = annual_snapshots.groupby(GROUP_COLUMN)["YEAR"].shift()
annual_snapshots["Previous_annual_score"] = annual_snapshots.groupby(GROUP_COLUMN)["MarketScore_hat"].shift()
annual_snapshots["Annual_score_change"] = (
    annual_snapshots["MarketScore_hat"] - annual_snapshots["Previous_annual_score"]
)
annual_changes = annual_snapshots.loc[
    annual_snapshots["YEAR"].sub(annual_snapshots["Previous_year"]).eq(1)
].copy()
annual_changes["Absolute_annual_score_change"] = annual_changes["Annual_score_change"].abs()
median_annual_score_change = annual_changes["Absolute_annual_score_change"].median()

zip_stability = (
    stability_data.dropna(subset=["Absolute_observation_change"])
    .groupby(GROUP_COLUMN)
    .agg(
        Transitions=("Absolute_observation_change", "size"),
        Maximum_observation_change=("Absolute_observation_change", "max"),
        Median_observation_change=("Absolute_observation_change", "median"),
    )
    .reset_index()
)
zip_stability["Has_large_jump"] = (
    zip_stability["Maximum_observation_change"] >= LARGE_JUMP_THRESHOLD
)
large_jump_zip_percentage = zip_stability["Has_large_jump"].mean() * 100
maximum_observation_change = stability_data["Absolute_observation_change"].max()

stability_summary = pd.DataFrame({
    "Measure": [
        "Median absolute annual score change",
        f"ZIPs with at least one >= {LARGE_JUMP_THRESHOLD}-point jump",
        "Maximum observation-to-observation change",
        "Consecutive annual ZIP pairs",
        "ZIPs eligible for jump analysis",
    ],
    "Value": [
        f"{median_annual_score_change:.1f} points",
        f"{large_jump_zip_percentage:.1f}%",
        f"{maximum_observation_change:.0f} points",
        f"{len(annual_changes):,}",
        f"{len(zip_stability):,}",
    ],
})
annual_change_by_year = (
    annual_changes.groupby("YEAR")["Absolute_annual_score_change"]
    .agg(Median="median", Mean="mean", Pairs="size")
    .reset_index()
)
largest_observation_changes = (
    stability_data.dropna(subset=["Absolute_observation_change"])
    .nlargest(10, "Absolute_observation_change")[[
        GROUP_COLUMN, "Previous_period", "PERIOD", "Previous_score",
        "MarketScore_hat", "Observation_change",
        "Absolute_observation_change", "Days_since_previous",
    ]]
)
display(stability_summary)
display(annual_change_by_year)
display(largest_observation_changes)

figure, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(
    annual_change_by_year["YEAR"], annual_change_by_year["Median"],
    color="#1B5E20", marker="o", linewidth=2.2,
)
axes[0].axhline(
    median_annual_score_change, color="#C44E00", linestyle="--",
    label=f"Overall median = {median_annual_score_change:.1f}",
)
axes[0].set(
    title="Median absolute annual score change", xlabel="Ending year",
    ylabel="Absolute score change (points)",
)
axes[0].legend(frameon=False)

largest_zip_changes = zip_stability.nlargest(20, "Maximum_observation_change").sort_values(
    "Maximum_observation_change"
)
bar_colors = np.where(
    largest_zip_changes["Maximum_observation_change"] >= LARGE_JUMP_THRESHOLD,
    "#C44E00", "#1B5E20",
)
axes[1].barh(
    largest_zip_changes[GROUP_COLUMN].astype(str),
    largest_zip_changes["Maximum_observation_change"], color=bar_colors,
)
axes[1].axvline(
    LARGE_JUMP_THRESHOLD, color="#171512", linestyle="--",
    label=f"Large-jump threshold = {LARGE_JUMP_THRESHOLD}",
)
axes[1].set(
    title="Largest observation-to-observation changes by ZIP",
    xlabel="Maximum absolute change (points)", ylabel="ZIP Code",
)
axes[1].legend(frameon=False)
for axis in axes:
    axis.grid(alpha=0.2)
figure.suptitle("Full-period Market Score stability diagnostics", fontsize=15)
figure.tight_layout()
stability_plot_path = plots_dir / "full_period_score_stability_diagnostics.png"
figure.savefig(stability_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved stability diagnostics to: {stability_plot_path}")

## additional sanity check
pick the top two variables that are most correlated with market score that are not incldued in the model, and validate if their relation with predicted market score are as expected (passing sign check)

In [ ]:
# Select the two strongest reference-score correlates not used by the champion model.
champion_variables = set(refit_lasso_selected_variables)
eligible_sanity_variables = [
    variable for variable in numerical_columns
    if variable in modeling_data.columns
    and variable not in champion_variables
    and sign_lookup_series.get(variable, "unconstrained") in {"positive", "negative"}
]
reference_correlation_rows = []
for variable in eligible_sanity_variables:
    valid = modeling_data[[variable, TARGET_COLUMN]].dropna()
    if len(valid) < 3 or valid[variable].nunique() < 2:
        continue
    correlation = float(spearmanr(valid[variable], valid[TARGET_COLUMN])[0])
    if np.isfinite(correlation):
        reference_correlation_rows.append({
            "Variable": variable,
            "Reference_Spearman": correlation,
            "Absolute_reference_Spearman": abs(correlation),
            "Expected_sign": sign_lookup_series[variable],
        })
reference_correlation_table = pd.DataFrame(reference_correlation_rows).sort_values(
    ["Absolute_reference_Spearman", "Variable"], ascending=[False, True]
)
top_two_external_signals = reference_correlation_table.head(2)["Variable"].tolist()
if len(top_two_external_signals) != 2:
    raise ValueError("Two eligible out-of-model sanity-check variables were not available.")

latest_sanity_data = latest_all_zip_data[[
    GROUP_COLUMN, "MarketScore_hat"
] + top_two_external_signals].copy()
sanity_check_rows = []
for variable in top_two_external_signals:
    valid = latest_sanity_data[[variable, "MarketScore_hat"]].dropna()
    predicted_correlation = float(spearmanr(
        valid[variable], valid["MarketScore_hat"]
    )[0])
    expected_sign = sign_lookup_series[variable]
    sign_pass = (
        (expected_sign == "positive" and predicted_correlation > 0)
        or (expected_sign == "negative" and predicted_correlation < 0)
    )
    reference_row = reference_correlation_table.loc[
        reference_correlation_table["Variable"].eq(variable)
    ].iloc[0]
    sanity_check_rows.append({
        "Variable": variable,
        "Expected_sign": expected_sign,
        "Reference_score_Spearman": reference_row["Reference_Spearman"],
        "Predicted_score_Spearman": predicted_correlation,
        "Rows": len(valid),
        "Sign_check": "PASS" if sign_pass else "FAIL",
    })
sanity_check_results = pd.DataFrame(sanity_check_rows)
display(sanity_check_results)

figure, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
for axis, result in zip(axes, sanity_check_rows):
    variable = result["Variable"]
    valid = latest_sanity_data[[variable, "MarketScore_hat"]].dropna()
    sns.regplot(
        data=valid, x=variable, y="MarketScore_hat", ax=axis,
        scatter_kws={"alpha": 0.45, "s": 28, "color": "#D97917"},
        line_kws={"color": "#1B5E20", "linewidth": 2},
    )
    axis.set(
        title=f"{variable}: {result['Sign_check']}",
        ylabel="MarketScore_hat (0-100)",
        ylim=(0, 100),
    )
    axis.text(
        0.04, 0.96,
        f"Expected: {result['Expected_sign']}\n"
        f"Spearman = {result['Predicted_score_Spearman']:.3f}",
        transform=axis.transAxes, va="top",
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )
    axis.grid(alpha=0.2)
figure.suptitle(
    "Out-of-model market-signal sanity check: all latest-period ZIPs", fontsize=15
)
figure.tight_layout()
sanity_plot_path = plots_dir / "latest_out_of_model_signal_sanity_check.png"
figure.savefig(sanity_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved out-of-model sanity check to: {sanity_plot_path}")

## Extreme jump analysis

Produce period-level feature contributions for ZIP 90021, 91313 and 92707, make plots of the predicted score and most significant contributing factor against period.

In [ ]:
# Decompose the champion's raw prediction into period-level feature contributions.
EXTREME_JUMP_ZIPS = [90021, 91313, 92070]
extreme_jump_data = (
    full_period_score_data.loc[
        full_period_score_data[GROUP_COLUMN].isin(EXTREME_JUMP_ZIPS)
    ]
    .sort_values([GROUP_COLUMN, "PERIOD_END"])
    .reset_index(drop=True)
)
missing_requested_zips = sorted(set(EXTREME_JUMP_ZIPS) - set(extreme_jump_data[GROUP_COLUMN]))
if missing_requested_zips:
    raise ValueError(f"Requested ZIPs are absent from full-period data: {missing_requested_zips}")

refit_preprocessor = refit_lasso_net_pipeline.named_steps["preprocessor"]
refit_model = refit_lasso_net_pipeline.named_steps["model"]
transformed_extreme_features = refit_preprocessor.transform(
    extreme_jump_data[refit_lasso_features]
)
encoded_feature_names = refit_lasso_coefficient_table["encoded_feature"].tolist()
if transformed_extreme_features.shape[1] != len(encoded_feature_names):
    raise ValueError("Transformed feature count does not match coefficient metadata.")
encoded_contributions = pd.DataFrame(
    transformed_extreme_features * refit_model.coef_,
    columns=encoded_feature_names, index=extreme_jump_data.index,
)
original_feature_contributions = pd.DataFrame(index=extreme_jump_data.index)
for variable in refit_lasso_coefficient_table["original_variable"].drop_duplicates():
    encoded_columns = refit_lasso_coefficient_table.loc[
        refit_lasso_coefficient_table["original_variable"].eq(variable),
        "encoded_feature",
    ].tolist()
    original_feature_contributions[variable] = encoded_contributions[encoded_columns].sum(axis=1)

extreme_jump_data["Raw_MarketScore_hat"] = refit_lasso_net_pipeline.predict(
    extreme_jump_data[refit_lasso_features]
)
extreme_jump_data["Contribution_sum_check"] = (
    refit_model.intercept_ + original_feature_contributions.sum(axis=1)
)
if not np.allclose(
    extreme_jump_data["Raw_MarketScore_hat"],
    extreme_jump_data["Contribution_sum_check"], atol=1e-8,
):
    raise ValueError("Feature contributions do not reconcile to raw model predictions.")

jump_summary_rows = []
period_contribution_tables = []
for zip_code in EXTREME_JUMP_ZIPS:
    zip_rows = extreme_jump_data.loc[
        extreme_jump_data[GROUP_COLUMN].eq(zip_code)
    ].copy()
    zip_contributions = original_feature_contributions.loc[zip_rows.index].copy()
    score_changes = zip_rows["MarketScore_hat"].diff().abs()
    if score_changes.notna().sum() == 0:
        continue
    jump_end_index = score_changes.idxmax()
    jump_position = zip_rows.index.get_loc(jump_end_index)
    jump_start_index = zip_rows.index[jump_position - 1]
    contribution_change = (
        zip_contributions.loc[jump_end_index] - zip_contributions.loc[jump_start_index]
    )
    jump_driver = contribution_change.abs().idxmax()
    jump_summary_rows.append({
        GROUP_COLUMN: zip_code,
        "From_PERIOD": extreme_jump_data.loc[jump_start_index, "PERIOD"],
        "To_PERIOD": extreme_jump_data.loc[jump_end_index, "PERIOD"],
        "From_score": extreme_jump_data.loc[jump_start_index, "MarketScore_hat"],
        "To_score": extreme_jump_data.loc[jump_end_index, "MarketScore_hat"],
        "Absolute_score_jump": score_changes.loc[jump_end_index],
        "Largest_contribution_change_driver": jump_driver,
        "Driver_contribution_before": zip_contributions.loc[jump_start_index, jump_driver],
        "Driver_contribution_after": zip_contributions.loc[jump_end_index, jump_driver],
        "Driver_contribution_change": contribution_change[jump_driver],
        "Driver_value_before": extreme_jump_data.loc[jump_start_index, jump_driver],
        "Driver_value_after": extreme_jump_data.loc[jump_end_index, jump_driver],
    })

    period_detail = zip_rows[[
        GROUP_COLUMN, "PERIOD", "MarketScore_hat", "Raw_MarketScore_hat", jump_driver
    ]].copy().rename(columns={jump_driver: "Selected_driver_value"})
    period_detail["Selected_driver"] = jump_driver
    period_detail["Selected_driver_contribution"] = zip_contributions[jump_driver].to_numpy()
    period_contribution_tables.append(period_detail)

    figure, score_axis = plt.subplots(figsize=(12, 5.5))
    contribution_axis = score_axis.twinx()
    score_axis.plot(
        zip_rows["PERIOD_END"], zip_rows["MarketScore_hat"],
        color="#1B5E20", marker="o", linewidth=2.4, label="MarketScore_hat",
    )
    contribution_axis.plot(
        zip_rows["PERIOD_END"], zip_contributions[jump_driver],
        color="#D97917", marker="s", linestyle=":", linewidth=2.0,
        label=f"{jump_driver} contribution",
    )
    for jump_index in [jump_start_index, jump_end_index]:
        score_axis.axvline(
            extreme_jump_data.loc[jump_index, "PERIOD_END"],
            color="#6B6258", linestyle="--", alpha=0.55,
        )
    score_axis.set(
        title=f"ZIP {zip_code}: score and largest extreme-jump contribution ({jump_driver})",
        xlabel="PERIOD", ylabel="MarketScore_hat (0-100)", ylim=(0, 100),
    )
    contribution_axis.set_ylabel("Feature contribution to raw score (points)")
    handles_1, labels_1 = score_axis.get_legend_handles_labels()
    handles_2, labels_2 = contribution_axis.get_legend_handles_labels()
    score_axis.legend(handles_1 + handles_2, labels_1 + labels_2, loc="best", frameon=False)
    score_axis.grid(alpha=0.2)
    figure.autofmt_xdate()
    figure.tight_layout()
    output_path = plots_dir / f"extreme_jump_ZIP_{zip_code}_score_contribution.png"
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

extreme_jump_summary = pd.DataFrame(jump_summary_rows)
extreme_jump_period_contributions = pd.concat(period_contribution_tables, ignore_index=True)
extreme_jump_period_contributions.to_csv(
    ROOT / "outputs" / "extreme_jump_feature_contributions.csv", index=False
)
display(extreme_jump_summary)
display(extreme_jump_period_contributions)

## Baseline Challenger

Since we find the Champion model have potention stability problems, lets perform the following check on baseline model and make the same plots as above.

1. Historical stability check 
2. additional sanity on top two explanatory variable outside of the baseline model


In [ ]:
# Score every historical row with the fixed three-variable baseline OLS model.
baseline_full_score_data = full_period_score_data.copy()
baseline_imputation_values = modeling_data[best_model_variables].median(numeric_only=True)
baseline_missing_rows = baseline_full_score_data[best_model_variables].isna().any(axis=1)
baseline_full_score_data[best_model_variables] = (
    baseline_full_score_data[best_model_variables].fillna(baseline_imputation_values)
)
baseline_full_design, _ = build_design_matrix(
    baseline_full_score_data, best_model_variables
)
baseline_full_design = baseline_full_design[best_fitted_model.model.exog_names]
baseline_full_raw_predictions = best_fitted_model.predict(baseline_full_design)
baseline_full_score_data["Baseline_MarketScore_hat"] = constrain_market_scores(
    baseline_full_raw_predictions, "Full-period baseline OLS"
)
print(
    f"Baseline median-imputed {baseline_missing_rows.sum():,} rows using values "
    "learned from the latest-period calibration sample."
)

# Repeat the champion stability diagnostics for the baseline challenger.
baseline_stability_data = baseline_full_score_data[[
    GROUP_COLUMN, "PERIOD", "PERIOD_END", "Baseline_MarketScore_hat"
]].copy()
baseline_stability_data["PERIOD_DATE"] = pd.to_datetime(
    baseline_stability_data["PERIOD_END"], errors="raise"
)
baseline_stability_data = baseline_stability_data.sort_values([GROUP_COLUMN, "PERIOD_DATE"])
baseline_stability_data["Previous_period"] = baseline_stability_data.groupby(GROUP_COLUMN)["PERIOD"].shift()
baseline_stability_data["Previous_score"] = baseline_stability_data.groupby(GROUP_COLUMN)["Baseline_MarketScore_hat"].shift()
baseline_stability_data["Observation_change"] = (
    baseline_stability_data["Baseline_MarketScore_hat"]
    - baseline_stability_data["Previous_score"]
)
baseline_stability_data["Absolute_observation_change"] = (
    baseline_stability_data["Observation_change"].abs()
)
baseline_stability_data["Days_since_previous"] = (
    baseline_stability_data.groupby(GROUP_COLUMN)["PERIOD_DATE"].diff().dt.days
)
baseline_stability_data["YEAR"] = baseline_stability_data["PERIOD_DATE"].dt.year
baseline_annual_snapshots = (
    baseline_stability_data.sort_values([GROUP_COLUMN, "YEAR", "PERIOD_DATE"])
    .groupby([GROUP_COLUMN, "YEAR"], as_index=False).tail(1)
    .sort_values([GROUP_COLUMN, "YEAR"])
)
baseline_annual_snapshots["Previous_year"] = baseline_annual_snapshots.groupby(GROUP_COLUMN)["YEAR"].shift()
baseline_annual_snapshots["Previous_annual_score"] = baseline_annual_snapshots.groupby(GROUP_COLUMN)["Baseline_MarketScore_hat"].shift()
baseline_annual_snapshots["Annual_score_change"] = (
    baseline_annual_snapshots["Baseline_MarketScore_hat"]
    - baseline_annual_snapshots["Previous_annual_score"]
)
baseline_annual_changes = baseline_annual_snapshots.loc[
    baseline_annual_snapshots["YEAR"].sub(
        baseline_annual_snapshots["Previous_year"]
    ).eq(1)
].copy()
baseline_annual_changes["Absolute_annual_score_change"] = (
    baseline_annual_changes["Annual_score_change"].abs()
)
baseline_annual_change_by_year = (
    baseline_annual_changes.groupby("YEAR")["Absolute_annual_score_change"]
    .agg(Median="median", Mean="mean", Pairs="size").reset_index()
)
baseline_zip_stability = (
    baseline_stability_data.dropna(subset=["Absolute_observation_change"])
    .groupby(GROUP_COLUMN)
    .agg(
        Transitions=("Absolute_observation_change", "size"),
        Maximum_observation_change=("Absolute_observation_change", "max"),
        Median_observation_change=("Absolute_observation_change", "median"),
    ).reset_index()
)
baseline_median_annual_change = baseline_annual_changes["Absolute_annual_score_change"].median()
stability_model_comparison = pd.DataFrame([
    {
        "Model": "Champion refitted LassoCV",
        "Median_absolute_annual_change": annual_changes["Absolute_annual_score_change"].median(),
        "ZIPs_with_20_point_jump_pct": (zip_stability["Maximum_observation_change"] >= 20).mean() * 100,
        "ZIPs_with_50_point_jump_pct": (zip_stability["Maximum_observation_change"] >= 50).mean() * 100,
        "Maximum_observation_change": zip_stability["Maximum_observation_change"].max(),
        "Boundary_score_pct": scores_all_periods["MarketScore_hat"].isin([0, 100]).mean() * 100,
    },
    {
        "Model": "Baseline OLS challenger",
        "Median_absolute_annual_change": baseline_median_annual_change,
        "ZIPs_with_20_point_jump_pct": (baseline_zip_stability["Maximum_observation_change"] >= 20).mean() * 100,
        "ZIPs_with_50_point_jump_pct": (baseline_zip_stability["Maximum_observation_change"] >= 50).mean() * 100,
        "Maximum_observation_change": baseline_zip_stability["Maximum_observation_change"].max(),
        "Boundary_score_pct": baseline_full_score_data["Baseline_MarketScore_hat"].isin([0, 100]).mean() * 100,
    },
])
display(stability_model_comparison)
display(
    baseline_stability_data.nlargest(10, "Absolute_observation_change")[[
        GROUP_COLUMN, "Previous_period", "PERIOD", "Previous_score",
        "Baseline_MarketScore_hat", "Observation_change",
        "Absolute_observation_change", "Days_since_previous",
    ]]
)

figure, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(
    annual_change_by_year["YEAR"], annual_change_by_year["Median"],
    color="#1B5E20", marker="o", linewidth=2.2, label="Champion LassoCV",
)
axes[0].plot(
    baseline_annual_change_by_year["YEAR"], baseline_annual_change_by_year["Median"],
    color="#D97917", marker="s", linewidth=2.2, label="Baseline OLS",
)
axes[0].set(
    title="Median absolute annual score change", xlabel="Ending year",
    ylabel="Absolute score change (points)",
)
axes[0].legend(frameon=False)
baseline_largest_zip_changes = baseline_zip_stability.nlargest(
    20, "Maximum_observation_change"
).sort_values("Maximum_observation_change")
axes[1].barh(
    baseline_largest_zip_changes[GROUP_COLUMN].astype(str),
    baseline_largest_zip_changes["Maximum_observation_change"], color="#D97917",
)
axes[1].axvline(20, color="#171512", linestyle="--", label="20-point threshold")
axes[1].axvline(50, color="#7B1E1E", linestyle=":", label="50-point threshold")
axes[1].set(
    title="Baseline: largest observation changes by ZIP",
    xlabel="Maximum absolute change (points)", ylabel="ZIP Code",
)
axes[1].legend(frameon=False)
for axis in axes:
    axis.grid(alpha=0.2)
figure.suptitle("Baseline OLS challenger stability diagnostics", fontsize=15)
figure.tight_layout()
baseline_stability_plot_path = plots_dir / "baseline_challenger_stability_diagnostics.png"
figure.savefig(baseline_stability_plot_path, dpi=300, bbox_inches="tight")
plt.show()

# Select and test the two strongest reference-score correlates outside the baseline.
baseline_external_candidates = [
    variable for variable in numerical_columns
    if variable in modeling_data.columns
    and variable not in set(best_model_variables)
    and sign_lookup_series.get(variable, "unconstrained") in {"positive", "negative"}
]
baseline_reference_rows = []
for variable in baseline_external_candidates:
    valid = modeling_data[[variable, TARGET_COLUMN]].dropna()
    if len(valid) < 3 or valid[variable].nunique() < 2:
        continue
    correlation = float(spearmanr(valid[variable], valid[TARGET_COLUMN])[0])
    if np.isfinite(correlation):
        baseline_reference_rows.append({
            "Variable": variable, "Reference_Spearman": correlation,
            "Absolute_reference_Spearman": abs(correlation),
            "Expected_sign": sign_lookup_series[variable],
        })
baseline_reference_correlations = pd.DataFrame(baseline_reference_rows).sort_values(
    ["Absolute_reference_Spearman", "Variable"], ascending=[False, True]
)
baseline_top_two_external = baseline_reference_correlations.head(2)["Variable"].tolist()
baseline_latest_data = baseline_full_score_data.loc[
    baseline_full_score_data["has_2025"].eq(True)
    & baseline_full_score_data["latest_period"].eq(True)
].copy()
baseline_sanity_rows = []
for variable in baseline_top_two_external:
    valid = baseline_latest_data[[variable, "Baseline_MarketScore_hat"]].dropna()
    predicted_correlation = float(spearmanr(
        valid[variable], valid["Baseline_MarketScore_hat"]
    )[0])
    expected_sign = sign_lookup_series[variable]
    sign_pass = (
        (expected_sign == "positive" and predicted_correlation > 0)
        or (expected_sign == "negative" and predicted_correlation < 0)
    )
    baseline_sanity_rows.append({
        "Variable": variable, "Expected_sign": expected_sign,
        "Reference_score_Spearman": baseline_reference_correlations.loc[
            baseline_reference_correlations["Variable"].eq(variable),
            "Reference_Spearman",
        ].iloc[0],
        "Predicted_score_Spearman": predicted_correlation,
        "Rows": len(valid), "Sign_check": "PASS" if sign_pass else "FAIL",
    })
baseline_sanity_results = pd.DataFrame(baseline_sanity_rows)
display(baseline_sanity_results)

figure, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
for axis, result in zip(axes, baseline_sanity_rows):
    variable = result["Variable"]
    valid = baseline_latest_data[[variable, "Baseline_MarketScore_hat"]].dropna()
    sns.regplot(
        data=valid, x=variable, y="Baseline_MarketScore_hat", ax=axis,
        scatter_kws={"alpha": 0.45, "s": 28, "color": "#D97917"},
        line_kws={"color": "#1B5E20", "linewidth": 2},
    )
    axis.set(
        title=f"{variable}: {result['Sign_check']}",
        ylabel="Baseline MarketScore_hat (0-100)", ylim=(0, 100),
    )
    axis.text(
        0.04, 0.96,
        f"Expected: {result['Expected_sign']}\n"
        f"Spearman = {result['Predicted_score_Spearman']:.3f}",
        transform=axis.transAxes, va="top",
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )
    axis.grid(alpha=0.2)
figure.suptitle("Baseline out-of-model signal sanity check: all latest-period ZIPs", fontsize=15)
figure.tight_layout()
baseline_sanity_plot_path = plots_dir / "baseline_latest_out_of_model_signal_sanity_check.png"
figure.savefig(baseline_sanity_plot_path, dpi=300, bbox_inches="tight")
plt.show()

### A Note

This result suggests that reducing the model to three variables does not solve the stability problem; predictor outliers, extrapolation, irregular time gaps, and boundary clipping remain more likely causes.

Consider the following solution to stablize the scores at historical period:
1. Apply transformation on highly concentrated MOM/YOY variables with long tails.
2. Analyze ditribution shift in historical period, and adjust the shifted variables accordingly.

## Distribution-shift analysis

Compare the 83 latest-period reference ZIPs used for model calibration with every row in the full-period scoring population. The analysis covers the champion score and every explanatory variable selected by the refitted LassoCV model.

The distribution plots combine empirical histograms and density estimates with fitted normal curves as descriptive references. The ECDF plots and quantitative statistics do not require a normal-distribution assumption. Missing observations are excluded separately for each variable rather than being imputed solely for visualization.

### Step 1 — Define comparison populations

The reference population contains the aligned latest-2025 rows with observed reference scores. Its `MarketScore_hat` values are the champion model's fitted latest-period predictions. The scoring population contains all existing ZIP-period rows and their champion full-period predictions.

In [ ]:
# Build comparable reference-calibration and full-period scoring populations.
selected_distribution_features = list(dict.fromkeys(refit_lasso_selected_variables))
missing_distribution_features = sorted(
    set(selected_distribution_features)
    - set(modeling_data.columns).intersection(full_period_score_data.columns)
)
if missing_distribution_features:
    raise ValueError(
        f"Selected features are unavailable for distribution analysis: "
        f"{missing_distribution_features}"
    )
distribution_shift_variables = ["MarketScore_hat"] + selected_distribution_features
percentage_distribution_variables = {
    variable for variable in selected_distribution_features
    if variable.endswith(("_MOM", "_YOY"))
    or variable in {
        "AVG_SALE_TO_LIST", "SOLD_ABOVE_LIST",
        "OFF_MARKET_IN_TWO_WEEKS",
    }
}
reference_distribution_data = modeling_data[[
    GROUP_COLUMN
] + selected_distribution_features].copy()
reference_distribution_data["MarketScore_hat"] = market_score_hat[
    "Market_Score_hat_latest"
].to_numpy()
full_period_distribution_data = full_period_score_data[[
    GROUP_COLUMN, "PERIOD", "MarketScore_hat"
] + selected_distribution_features].copy()
distribution_population_summary = pd.DataFrame({
    "Population": ["Reference calibration ZIPs", "Full-period scoring rows"],
    "Rows": [len(reference_distribution_data), len(full_period_distribution_data)],
    "ZIP_codes": [
        reference_distribution_data[GROUP_COLUMN].nunique(),
        full_period_distribution_data[GROUP_COLUMN].nunique(),
    ],
})
display(distribution_population_summary)

### Step 2 — Quantify distribution differences

For each variable, compare medians and tail percentiles, measure how often full-period observations fall outside the reference 1st–99th percentile interval, and calculate the Kolmogorov–Smirnov and Wasserstein distances. The KS statistic measures the maximum separation between the two empirical cumulative distributions; the Wasserstein distance measures their average separation in the variable's original units.

In [ ]:
from scipy.stats import ks_2samp, norm, wasserstein_distance

distribution_shift_rows = []
for variable in distribution_shift_variables:
    reference_values = pd.to_numeric(
        reference_distribution_data[variable], errors="coerce"
    ).dropna()
    full_values = pd.to_numeric(
        full_period_distribution_data[variable], errors="coerce"
    ).dropna()
    reference_p01, reference_p99 = reference_values.quantile([0.01, 0.99])
    full_p01, full_p99 = full_values.quantile([0.01, 0.99])
    # ks_result = ks_2samp(reference_values, full_values, method="auto")
    try:
        ks_result = ks_2samp(reference_values, full_values, method="auto")
    except TypeError:
        ks_result = ks_2samp(reference_values, full_values)
    distribution_shift_rows.append({
        "Variable": variable,
        "Reference_rows": len(reference_values),
        "Full_period_rows": len(full_values),
        "Reference_median": reference_values.median(),
        "Full_period_median": full_values.median(),
        "Reference_P01": reference_p01,
        "Reference_P99": reference_p99,
        "Full_period_P01": full_p01,
        "Full_period_P99": full_p99,
        "Full_period_below_reference_P01_pct": (full_values < reference_p01).mean() * 100,
        "Full_period_above_reference_P99_pct": (full_values > reference_p99).mean() * 100,
        "KS_statistic": float(ks_result[0]),
        "KS_p_value": float(ks_result[1]),
        "Wasserstein_distance": float(wasserstein_distance(reference_values, full_values)),
    })
distribution_shift_summary = pd.DataFrame(distribution_shift_rows)
distribution_shift_output_path = ROOT / "outputs" / "distribution_shift_summary.csv"
distribution_shift_summary.to_csv(distribution_shift_output_path, index=False)
display(distribution_shift_summary)
print(f"Saved distribution-shift summary to: {distribution_shift_output_path}")

### Step 3 — Compare empirical distributions

Each figure overlays density-normalized empirical histograms and KDE curves for the two populations. Dashed normal-density curves are included only as visual references. Vertical lines identify population medians, and the shaded band shows the reference population's 1st–99th percentile support. For readability, the displayed x-range is trimmed to the pooled 0.5th–99.5th percentiles when extreme tails would otherwise compress the central distribution; all summary calculations continue to use the complete data.

In [ ]:
from matplotlib.ticker import PercentFormatter

REFERENCE_COLOR = "#1B5E20"
FULL_PERIOD_COLOR = "#D97917"
distribution_plot_paths = {}
for variable in distribution_shift_variables:
    reference_values = pd.to_numeric(
        reference_distribution_data[variable], errors="coerce"
    ).dropna()
    full_values = pd.to_numeric(
        full_period_distribution_data[variable], errors="coerce"
    ).dropna()
    pooled_values = pd.concat([reference_values, full_values], ignore_index=True)
    reference_p01, reference_p99 = reference_values.quantile([0.01, 0.99])

    if variable == "MarketScore_hat":
        histogram_bins = np.arange(-0.5, 101.5, 2)
        display_minimum, display_maximum = 0, 100
    else:
        display_minimum, display_maximum = pooled_values.quantile([0.005, 0.995])
        histogram_bins = np.linspace(display_minimum, display_maximum, 45)
    trimmed_reference = reference_values.between(display_minimum, display_maximum)
    trimmed_full = full_values.between(display_minimum, display_maximum)

    figure, axis = plt.subplots(figsize=(11, 6))
    axis.axvspan(
        max(reference_p01, display_minimum), min(reference_p99, display_maximum),
        color=REFERENCE_COLOR, alpha=0.08, label="Reference P01-P99 support",
    )
    axis.hist(
        reference_values[trimmed_reference], bins=histogram_bins, density=True,
        alpha=0.28, color=REFERENCE_COLOR, label="Reference empirical histogram",
    )
    axis.hist(
        full_values[trimmed_full], bins=histogram_bins, density=True,
        alpha=0.25, color=FULL_PERIOD_COLOR, label="Full-period empirical histogram",
    )
    if reference_values.nunique() >= 5 and reference_values.std(ddof=0) > 0:
        sns.kdeplot(
            x=reference_values, ax=axis, color=REFERENCE_COLOR, linewidth=2.2,
            label="Reference KDE", clip=(display_minimum, display_maximum),
        )
    if full_values.nunique() >= 5 and full_values.std(ddof=0) > 0:
        sns.kdeplot(
            x=full_values, ax=axis, color=FULL_PERIOD_COLOR, linewidth=2.2,
            label="Full-period KDE", clip=(display_minimum, display_maximum),
        )
    density_grid = np.linspace(display_minimum, display_maximum, 500)
    for values, color, label in [
        (reference_values, REFERENCE_COLOR, "Reference fitted normal"),
        (full_values, FULL_PERIOD_COLOR, "Full-period fitted normal"),
    ]:
        standard_deviation = values.std(ddof=0)
        if standard_deviation > 0:
            axis.plot(
                density_grid, norm.pdf(density_grid, values.mean(), standard_deviation),
                color=color, linestyle="--", linewidth=1.5, alpha=0.9, label=label,
            )
    axis.axvline(reference_values.median(), color=REFERENCE_COLOR, linestyle=":", linewidth=2)
    axis.axvline(full_values.median(), color=FULL_PERIOD_COLOR, linestyle=":", linewidth=2)
    axis.set(
        title=f"Distribution shift: {variable}", xlabel=variable,
        ylabel="Density", xlim=(display_minimum, display_maximum),
    )
    if variable in percentage_distribution_variables:
        axis.xaxis.set_major_formatter(PercentFormatter(1.0))
    trimmed_reference_count = int((~trimmed_reference).sum())
    trimmed_full_count = int((~trimmed_full).sum())
    axis.text(
        0.98, 0.97,
        f"Reference n = {len(reference_values):,}\nFull-period n = {len(full_values):,}\n"
        f"Outside display range: {trimmed_reference_count:,} / {trimmed_full_count:,}",
        transform=axis.transAxes, ha="right", va="top",
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )
    axis.legend(loc="best", fontsize=8, frameon=False)
    axis.grid(alpha=0.18)
    figure.tight_layout()
    output_path = plots_dir / f"distribution_shift_{variable}.png"
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    distribution_plot_paths[variable] = output_path
    plt.show()

### Step 4 — Compare empirical cumulative distributions

ECDFs show the entire observed distribution without selecting histogram bins or assuming a theoretical distribution. Greater vertical separation indicates a larger distribution difference.

In [ ]:
ecdf_plot_paths = {}
for variable in distribution_shift_variables:
    reference_values = pd.to_numeric(
        reference_distribution_data[variable], errors="coerce"
    ).dropna()
    full_values = pd.to_numeric(
        full_period_distribution_data[variable], errors="coerce"
    ).dropna()
    figure, axis = plt.subplots(figsize=(10, 5.5))
    sns.ecdfplot(
        x=reference_values, ax=axis, color=REFERENCE_COLOR, linewidth=2.4,
        label=f"Reference calibration ZIPs (n={len(reference_values):,})",
    )
    sns.ecdfplot(
        x=full_values, ax=axis, color=FULL_PERIOD_COLOR, linewidth=2.2,
        label=f"Full-period scoring rows (n={len(full_values):,})",
    )
    axis.set(
        title=f"ECDF distribution comparison: {variable}",
        xlabel=variable, ylabel="Cumulative probability", ylim=(0, 1),
    )
    if variable in percentage_distribution_variables:
        axis.xaxis.set_major_formatter(PercentFormatter(1.0))
    if variable == "MEDIAN_DOM":
        axis.set_xscale("symlog", linthresh=10)
        axis.set_xlim(left=0)
    axis.legend(loc="best", frameon=False)
    axis.grid(alpha=0.2)
    figure.tight_layout()
    output_path = plots_dir / f"ecdf_shift_{variable}.png"
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    ecdf_plot_paths[variable] = output_path
    plt.show()

### Interpretation safeguards

- The fitted normal curves are descriptive overlays, not an assumption that these variables are normally distributed. The empirical histograms, KDEs, ECDFs, and quantiles should drive interpretation.
- With more than 12,000 full-period rows, a small difference can produce a statistically significant KS p-value. Emphasize the KS statistic, Wasserstein distance, percentile movement, and calibration-support violations rather than the p-value alone.
- The comparison combines temporal change, geographic composition differences, and possible data-quality errors. Distribution separation alone does not prove that an extreme observation is incorrect.
- `MarketScore_hat` is a model output, whereas the selected explanatory variables are model inputs. A shifted score distribution describes changed model behavior; shifted input distributions identify possible extrapolation or data-quality risk that may be causing that behavior.

# Full-period Scoring (Adjusted)

The purpose of this section is to adjust the distribution shift in full period data and achieve a stable historical score hat.


## Extreme value triming
1. make a copy of full_period_data and name it full_period_score_data_adjusted; this data will be used to adjust distribution shift.
2. for explanatory variable in full period dataset, cap or floor all outliers (range below Q₁ - 1.5 × IQR or above Q₃ + 1.5 × IQR of the reference dataset) to the range limits.
3. regenerate full period scoring and name the dataset scores_all_periods_adjusted
4. regenerate the analysis and plots to show historical stability and perform additional sanity check as in previous section.

In [ ]:
# Derive extreme-value limits from the latest-period reference sample only.
adjusted_explanatory_variables = [
    variable for variable in refit_lasso_selected_variables
    if variable in full_period_data.columns
    and variable in modeling_data.columns
    and pd.api.types.is_numeric_dtype(modeling_data[variable])
]
if not adjusted_explanatory_variables:
    raise ValueError("No numerical champion explanatory variables are available for trimming.")

iqr_limit_rows = []
for variable in adjusted_explanatory_variables:
    reference_values = pd.to_numeric(modeling_data[variable], errors="coerce").dropna()
    q1, q3 = reference_values.quantile([0.25, 0.75])
    iqr = q3 - q1
    iqr_limit_rows.append({
        "Variable": variable, "Reference_Q1": q1, "Reference_Q3": q3,
        "Reference_IQR": iqr, "Lower_limit": q1 - 1.5 * iqr,
        "Upper_limit": q3 + 1.5 * iqr,
    })
iqr_limits_adjusted = pd.DataFrame(iqr_limit_rows).set_index("Variable")
display(iqr_limits_adjusted.reset_index())

In [ ]:
# Copy the full-period data and winsorize selected model inputs to reference IQR limits.
full_period_score_data_adjusted = full_period_data.copy()
trim_summary_rows = []
for variable, limits in iqr_limits_adjusted.iterrows():
    values = pd.to_numeric(full_period_score_data_adjusted[variable], errors="coerce")
    below = values.lt(limits["Lower_limit"])
    above = values.gt(limits["Upper_limit"])
    full_period_score_data_adjusted[variable] = values.clip(
        lower=limits["Lower_limit"], upper=limits["Upper_limit"]
    )
    trim_summary_rows.append({
        "Variable": variable, "Floored_rows": int(below.sum()),
        "Capped_rows": int(above.sum()),
        "Adjusted_rows": int((below | above).sum()),
        "Adjusted_pct": (below | above).mean() * 100,
    })
trim_summary_adjusted = pd.DataFrame(trim_summary_rows).sort_values(
    ["Adjusted_rows", "Variable"], ascending=[False, True]
)
display(trim_summary_adjusted)

In [ ]:
# Score every adjusted row with the unchanged champion pipeline.
full_period_score_data_adjusted["PERIOD_END"] = pd.to_datetime(
    full_period_score_data_adjusted["PERIOD_END"], errors="raise"
)
full_period_score_data_adjusted["PERIOD"] = (
    full_period_score_data_adjusted["PERIOD_END"].dt.to_period("M").astype(str)
)
full_period_score_data_adjusted["City"] = (
    full_period_score_data_adjusted[GROUP_COLUMN].map(city_lookup)
)
adjusted_raw_predictions = refit_lasso_net_pipeline.predict(
    full_period_score_data_adjusted[refit_lasso_features]
)
full_period_score_data_adjusted["MarketScore_hat"] = constrain_market_scores(
    adjusted_raw_predictions, "IQR-adjusted full-period refitted LassoCV"
)
scores_all_periods_adjusted = (
    full_period_score_data_adjusted[[
        GROUP_COLUMN, "City", "County", "PERIOD", "MarketScore_hat"
    ]].sort_values([GROUP_COLUMN, "PERIOD"]).reset_index(drop=True)
)
if scores_all_periods_adjusted.duplicated([GROUP_COLUMN, "PERIOD"]).any():
    raise ValueError("Adjusted output contains duplicate (Zip Code, PERIOD) keys.")
if not scores_all_periods_adjusted["MarketScore_hat"].between(0, 100).all():
    raise ValueError("Adjusted scores fall outside 0-100.")
scores_all_periods_adjusted_path = score_output_dir / "scores_all_periods_adjusted.csv"
scores_all_periods_adjusted.to_csv(scores_all_periods_adjusted_path, index=False)
display(pd.DataFrame([{
    "Rows_scored": len(scores_all_periods_adjusted),
    "ZIP_codes": scores_all_periods_adjusted[GROUP_COLUMN].nunique(),
    "Floored_at_0": int((adjusted_raw_predictions < 0).sum()),
    "Capped_at_100": int((adjusted_raw_predictions > 100).sum()),
}]))
print(f"Saved adjusted full-period scores to: {scores_all_periods_adjusted_path}")

In [ ]:
# Recalculate the historical stability diagnostics on adjusted scores.
adjusted_stability_data = scores_all_periods_adjusted.copy()
adjusted_stability_data["PERIOD_DATE"] = pd.to_datetime(
    adjusted_stability_data["PERIOD"], format="%Y-%m", errors="raise"
)
adjusted_stability_data = adjusted_stability_data.sort_values([GROUP_COLUMN, "PERIOD_DATE"])
adjusted_stability_data["Previous_period"] = adjusted_stability_data.groupby(GROUP_COLUMN)["PERIOD"].shift()
adjusted_stability_data["Previous_score"] = adjusted_stability_data.groupby(GROUP_COLUMN)["MarketScore_hat"].shift()
adjusted_stability_data["Observation_change"] = adjusted_stability_data["MarketScore_hat"] - adjusted_stability_data["Previous_score"]
adjusted_stability_data["Absolute_observation_change"] = adjusted_stability_data["Observation_change"].abs()
adjusted_stability_data["Days_since_previous"] = adjusted_stability_data.groupby(GROUP_COLUMN)["PERIOD_DATE"].diff().dt.days
adjusted_stability_data["YEAR"] = adjusted_stability_data["PERIOD_DATE"].dt.year
adjusted_annual_snapshots = (
    adjusted_stability_data.sort_values([GROUP_COLUMN, "YEAR", "PERIOD_DATE"])
    .groupby([GROUP_COLUMN, "YEAR"], as_index=False).tail(1)
    .sort_values([GROUP_COLUMN, "YEAR"])
)
adjusted_annual_snapshots["Previous_year"] = adjusted_annual_snapshots.groupby(GROUP_COLUMN)["YEAR"].shift()
adjusted_annual_snapshots["Previous_annual_score"] = adjusted_annual_snapshots.groupby(GROUP_COLUMN)["MarketScore_hat"].shift()
adjusted_annual_snapshots["Annual_score_change"] = adjusted_annual_snapshots["MarketScore_hat"] - adjusted_annual_snapshots["Previous_annual_score"]
adjusted_annual_changes = adjusted_annual_snapshots.loc[
    adjusted_annual_snapshots["YEAR"].sub(adjusted_annual_snapshots["Previous_year"]).eq(1)
].copy()
adjusted_annual_changes["Absolute_annual_score_change"] = adjusted_annual_changes["Annual_score_change"].abs()
adjusted_annual_change_by_year = (
    adjusted_annual_changes.groupby("YEAR")["Absolute_annual_score_change"]
    .agg(Median="median", Mean="mean", Pairs="size").reset_index()
)
adjusted_zip_stability = (
    adjusted_stability_data.dropna(subset=["Absolute_observation_change"])
    .groupby(GROUP_COLUMN).agg(
        Transitions=("Absolute_observation_change", "size"),
        Maximum_observation_change=("Absolute_observation_change", "max"),
        Median_observation_change=("Absolute_observation_change", "median"),
    ).reset_index()
)
adjusted_stability_summary = pd.DataFrame([{
    "Model": "IQR-adjusted refitted LassoCV",
    "Median_absolute_annual_change": adjusted_annual_changes["Absolute_annual_score_change"].median(),
    "ZIPs_with_20_point_jump_pct": adjusted_zip_stability["Maximum_observation_change"].ge(20).mean() * 100,
    "ZIPs_with_50_point_jump_pct": adjusted_zip_stability["Maximum_observation_change"].ge(LARGE_JUMP_THRESHOLD).mean() * 100,
    "Maximum_observation_change": adjusted_stability_data["Absolute_observation_change"].max(),
    "Boundary_score_pct": adjusted_stability_data["MarketScore_hat"].isin([0, 100]).mean() * 100,
}])
display(adjusted_stability_summary)
display(adjusted_annual_change_by_year)
display(adjusted_stability_data.nlargest(10, "Absolute_observation_change")[[
    GROUP_COLUMN, "Previous_period", "PERIOD", "Previous_score",
    "MarketScore_hat", "Observation_change", "Absolute_observation_change",
    "Days_since_previous",
]])

In [ ]:
# Compare original and IQR-adjusted champion stability and save the adjusted diagnostic plot.
original_champion_summary = pd.DataFrame([{
    "Model": "Original refitted LassoCV",
    "Median_absolute_annual_change": annual_changes["Absolute_annual_score_change"].median(),
    "ZIPs_with_20_point_jump_pct": zip_stability["Maximum_observation_change"].ge(20).mean() * 100,
    "ZIPs_with_50_point_jump_pct": zip_stability["Maximum_observation_change"].ge(LARGE_JUMP_THRESHOLD).mean() * 100,
    "Maximum_observation_change": stability_data["Absolute_observation_change"].max(),
    "Boundary_score_pct": stability_data["MarketScore_hat"].isin([0, 100]).mean() * 100,
}])
adjusted_stability_comparison = pd.concat(
    [original_champion_summary, adjusted_stability_summary], ignore_index=True
)
display(adjusted_stability_comparison)

figure, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(annual_change_by_year["YEAR"], annual_change_by_year["Median"], color="#6B7280", marker="o", linewidth=2, label="Original")
axes[0].plot(adjusted_annual_change_by_year["YEAR"], adjusted_annual_change_by_year["Median"], color="#1B5E20", marker="o", linewidth=2.2, label="IQR adjusted")
axes[0].set(title="Median absolute annual score change", xlabel="Ending year", ylabel="Absolute score change (points)")
axes[0].legend(frameon=False)
adjusted_largest_zip_changes = adjusted_zip_stability.nlargest(20, "Maximum_observation_change").sort_values("Maximum_observation_change")
axes[1].barh(adjusted_largest_zip_changes[GROUP_COLUMN].astype(str), adjusted_largest_zip_changes["Maximum_observation_change"], color=np.where(adjusted_largest_zip_changes["Maximum_observation_change"].ge(LARGE_JUMP_THRESHOLD), "#C44E00", "#1B5E20"))
axes[1].axvline(LARGE_JUMP_THRESHOLD, color="#171512", linestyle="--", label=f"Large-jump threshold = {LARGE_JUMP_THRESHOLD}")
axes[1].set(title="Adjusted: largest observation changes by ZIP", xlabel="Maximum absolute change (points)", ylabel="ZIP Code")
axes[1].legend(frameon=False)
for axis in axes: axis.grid(alpha=0.2)
figure.suptitle("IQR-adjusted full-period score stability", fontsize=15)
figure.tight_layout()
adjusted_stability_plot_path = plots_dir / "full_period_score_stability_adjusted_diagnostics.png"
figure.savefig(adjusted_stability_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved adjusted stability diagnostics to: {adjusted_stability_plot_path}")

In [ ]:
# Repeat the out-of-model directional sanity check on adjusted latest-period scores.
required_adjusted_flags = {"has_2025", "latest_period"}
if not required_adjusted_flags.issubset(full_period_score_data_adjusted.columns):
    raise ValueError("Adjusted data is missing has_2025 or latest_period.")
adjusted_latest_sanity_data = full_period_score_data_adjusted.loc[
    full_period_score_data_adjusted["has_2025"].eq(True)
    & full_period_score_data_adjusted["latest_period"].eq(True),
    [GROUP_COLUMN, "MarketScore_hat"] + top_two_external_signals,
].copy()
adjusted_sanity_rows = []
for variable in top_two_external_signals:
    valid = adjusted_latest_sanity_data[[variable, "MarketScore_hat"]].dropna()
    expected_sign = sign_lookup_series[variable]
    testable = len(valid) >= 3 and valid[variable].nunique() >= 2 and valid["MarketScore_hat"].nunique() >= 2
    predicted_correlation = float(spearmanr(valid[variable], valid["MarketScore_hat"])[0]) if testable else np.nan
    sign_pass = (
        (expected_sign == "positive" and predicted_correlation > 0)
        or (expected_sign == "negative" and predicted_correlation < 0)
    ) if np.isfinite(predicted_correlation) else False
    reference_row = reference_correlation_table.loc[
        reference_correlation_table["Variable"].eq(variable)
    ].iloc[0]
    adjusted_sanity_rows.append({
        "Variable": variable, "Expected_sign": expected_sign,
        "Reference_score_Spearman": reference_row["Reference_Spearman"],
        "Adjusted_predicted_score_Spearman": predicted_correlation,
        "Rows": len(valid),
        "Sign_check": "PASS" if sign_pass else "NOT TESTABLE" if not testable else "FAIL",
    })
adjusted_sanity_check_results = pd.DataFrame(adjusted_sanity_rows)
display(adjusted_sanity_check_results)

In [ ]:
# Visualize the adjusted score's relationship with the same two independent signals.
figure, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
for axis, result in zip(axes, adjusted_sanity_rows):
    variable = result["Variable"]
    valid = adjusted_latest_sanity_data[[variable, "MarketScore_hat"]].dropna()
    sns.regplot(data=valid, x=variable, y="MarketScore_hat", ax=axis, scatter_kws={"alpha": 0.45, "s": 28, "color": "#D97917"}, line_kws={"color": "#1B5E20", "linewidth": 2})
    correlation_text = (f"{result['Adjusted_predicted_score_Spearman']:.3f}" if np.isfinite(result["Adjusted_predicted_score_Spearman"]) else "N/A")
    axis.set(title=f"{variable}: {result['Sign_check']}", ylabel="Adjusted MarketScore_hat (0-100)", ylim=(0, 100))
    axis.text(0.04, 0.96, f"Expected: {result['Expected_sign']}\nSpearman = {correlation_text}", transform=axis.transAxes, va="top", bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"})
    axis.grid(alpha=0.2)
figure.suptitle("IQR-adjusted score: out-of-model market-signal sanity check", fontsize=15)
figure.tight_layout()
adjusted_sanity_plot_path = plots_dir / "adjusted_latest_out_of_model_signal_sanity_check.png"
figure.savefig(adjusted_sanity_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved adjusted sanity check to: {adjusted_sanity_plot_path}")

In [ ]:
# Compare adjusted full-period predictions with the fixed latest-period reference distribution.
adjusted_score_values = pd.to_numeric(
    scores_all_periods_adjusted["MarketScore_hat"], errors="coerce"
).dropna()
reference_score_values = pd.to_numeric(
    reference_distribution_data["MarketScore_hat"], errors="coerce"
).dropna()
if len(adjusted_score_values) < 2 or len(reference_score_values) < 2:
    raise ValueError("Both score populations require at least two observations.")

score_bins = np.arange(-0.5, 101.5, 2)
score_grid = np.linspace(0, 100, 500)
figure, axis = plt.subplots(figsize=(11, 6))
axis.hist(reference_score_values, bins=score_bins, density=True, alpha=0.28, color=REFERENCE_COLOR, label=f"Latest-period reference (n={len(reference_score_values):,})")
axis.hist(adjusted_score_values, bins=score_bins, density=True, alpha=0.22, color=FULL_PERIOD_COLOR, label=f"Adjusted full period (n={len(adjusted_score_values):,})")
sns.kdeplot(x=reference_score_values, ax=axis, color=REFERENCE_COLOR, linewidth=2.4, clip=(0, 100), label="Reference KDE")
sns.kdeplot(x=adjusted_score_values, ax=axis, color=FULL_PERIOD_COLOR, linewidth=2.2, clip=(0, 100), label="Adjusted KDE")
for values, color, label in [(reference_score_values, REFERENCE_COLOR, "Reference fitted normal"), (adjusted_score_values, FULL_PERIOD_COLOR, "Adjusted fitted normal")]:
    mean_value, standard_deviation = values.mean(), values.std(ddof=1)
    if np.isfinite(standard_deviation) and standard_deviation > 0:
        axis.plot(score_grid, norm.pdf(score_grid, mean_value, standard_deviation), color=color, linestyle="--", linewidth=1.8, label=label)
    axis.axvline(values.median(), color=color, linestyle=":", linewidth=1.8)
axis.set(title="Adjusted full-period score distribution vs latest-period reference", xlabel="MarketScore_hat", ylabel="Density", xlim=(0, 100))
axis.grid(alpha=0.2)
axis.legend(frameon=False)
figure.tight_layout()
adjusted_score_distribution_plot_path = plots_dir / "adjusted_distribution_shift_MarketScore_hat.png"
figure.savefig(adjusted_score_distribution_plot_path, dpi=300, bbox_inches="tight")
plt.show()

figure, axis = plt.subplots(figsize=(10, 5.5))
sns.ecdfplot(x=reference_score_values, ax=axis, color=REFERENCE_COLOR, linewidth=2.4, label=f"Latest-period reference (n={len(reference_score_values):,})")
sns.ecdfplot(x=adjusted_score_values, ax=axis, color=FULL_PERIOD_COLOR, linewidth=2.2, label=f"Adjusted full period (n={len(adjusted_score_values):,})")
axis.set(title="Adjusted full-period score ECDF vs latest-period reference", xlabel="MarketScore_hat", ylabel="Cumulative probability", xlim=(0, 100), ylim=(0, 1))
axis.grid(alpha=0.2)
axis.legend(frameon=False)
figure.tight_layout()
adjusted_score_ecdf_plot_path = plots_dir / "adjusted_ecdf_shift_MarketScore_hat.png"
figure.savefig(adjusted_score_ecdf_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved adjusted score distribution plots to: {adjusted_score_distribution_plot_path} and {adjusted_score_ecdf_plot_path}")

## Distribution Adjustment Overlay

Adjust the distribution of market score hat in full period through quantile mapping.

Consider:

- **A**: latest-period data
- **B**: full-period data

### Transform B → A

Rather than directly mapping B to the empirical distribution of A, assume that
the latest-period data follows a Normal distribution:

$$
Y_A \sim \mathcal{N}(\mu_A,\sigma_A^2),
$$

where $\mu_A$ and $\sigma_A$ are estimated from the latest-period sample A.

For the full-period data:

$$
Y_B \sim F_B.
$$

First, map each observation in B to its percentile under the distribution of B:

$$
U_B = F_B(Y_B).
$$

Then map this percentile to the fitted Normal distribution of A:

$$
Y'_B
=
F_{A,\mathrm{Normal}}^{-1}(U_B)
=
\mu_A + \sigma_A \Phi^{-1}(U_B).
$$

Therefore,

$$
\boxed{
Y'_B
=
\mu_A + \sigma_A \Phi^{-1}\left(F_B(Y_B)\right)
}
$$

where $\Phi^{-1}$ is the inverse CDF (quantile function) of the standard
Normal distribution.

The transformed variable $Y'_B$ approximately follows:

$$
Y'_B \sim \mathcal{N}(\mu_A,\sigma_A^2).
$$

Plot the distribution comparison and ecdf comparison of market score hat against reference, and save the final dataset as scores_all_periods_adjusted.csv.

In [ ]:
# Map the IQR-trimmed full-period scores (B) to a Normal distribution fitted on A.
scores_all_periods_trimmed = scores_all_periods_adjusted.copy()
population_b = pd.to_numeric(
    scores_all_periods_trimmed["MarketScore_hat"], errors="coerce"
)
population_a = pd.to_numeric(
    reference_distribution_data["MarketScore_hat"], errors="coerce"
).dropna().sort_values().to_numpy()
if population_b.isna().any():
    raise ValueError("Full-period scores contain missing values before quantile mapping.")
if len(population_a) < 2 or np.unique(population_a).size < 2:
    raise ValueError("The latest-period reference distribution cannot support a Normal fit.")

reference_normal_mean = float(np.mean(population_a))
reference_normal_std = float(np.std(population_a, ddof=1))
if not np.isfinite(reference_normal_std) or reference_normal_std <= 0:
    raise ValueError("The fitted latest-period Normal distribution has invalid dispersion.")

# Midpoint empirical percentiles preserve ties and avoid 0/1 inputs to norm.ppf.
population_b_ranks = population_b.rank(method="average").to_numpy()
population_b_percentiles = (population_b_ranks - 0.5) / len(population_b)
quantile_mapped_raw = reference_normal_mean + reference_normal_std * norm.ppf(
    population_b_percentiles
)
quantile_mapped_scores = np.clip(np.rint(quantile_mapped_raw), 0, 100).astype(int)

scores_all_periods_adjusted = scores_all_periods_trimmed.copy()
scores_all_periods_adjusted["MarketScore_hat"] = quantile_mapped_scores
if not scores_all_periods_adjusted["MarketScore_hat"].between(0, 100).all():
    raise ValueError("Quantile-mapped scores fall outside 0-100.")
if scores_all_periods_adjusted.duplicated([GROUP_COLUMN, "PERIOD"]).any():
    raise ValueError("Quantile-mapped output contains duplicate (Zip Code, PERIOD) keys.")
scores_all_periods_adjusted.to_csv(scores_all_periods_adjusted_path, index=False)
quantile_mapping_summary = pd.DataFrame([{
    "Population": "Latest-period reference sample", "Rows": len(population_a),
    "Minimum": np.min(population_a), "Median": np.median(population_a),
    "Maximum": np.max(population_a),
}, {
    "Population": "Full period before quantile mapping", "Rows": len(population_b),
    "Minimum": population_b.min(), "Median": population_b.median(),
    "Maximum": population_b.max(),
}, {
    "Population": "Full period after fitted-Normal mapping", "Rows": len(quantile_mapped_scores),
    "Minimum": quantile_mapped_scores.min(), "Median": np.median(quantile_mapped_scores),
    "Maximum": quantile_mapped_scores.max(),
}])
display(quantile_mapping_summary)
display(pd.DataFrame([{"Fitted_Normal_mean": reference_normal_mean, "Fitted_Normal_std": reference_normal_std}]))
print(f"Saved final fitted-Normal-mapped scores to: {scores_all_periods_adjusted_path}")

In [ ]:
# Overlay the reference sample, its fitted Normal target, and scores before/after mapping.
mapped_score_values = scores_all_periods_adjusted["MarketScore_hat"].astype(float)
pre_mapping_score_values = scores_all_periods_trimmed["MarketScore_hat"].astype(float)
reference_overlay_values = pd.Series(population_a, dtype=float)
score_bins = np.arange(-0.5, 101.5, 2)
score_grid = np.linspace(0, 100, 500)
figure, axis = plt.subplots(figsize=(11, 6))
for values, color, alpha, label in [
    (reference_overlay_values, REFERENCE_COLOR, 0.28, "Latest-period reference"),
    (pre_mapping_score_values, "#6B7280", 0.14, "Before quantile mapping"),
    (mapped_score_values, FULL_PERIOD_COLOR, 0.22, "After fitted-Normal mapping"),
]:
    axis.hist(values, bins=score_bins, density=True, alpha=alpha, color=color, label=f"{label} (n={len(values):,})")
    sns.kdeplot(x=values, ax=axis, color=color, linewidth=2.2, clip=(0, 100), label=f"{label} KDE")
    axis.axvline(values.median(), color=color, linestyle=":", linewidth=1.6)
axis.plot(score_grid, norm.pdf(score_grid, reference_normal_mean, reference_normal_std), color=REFERENCE_COLOR, linestyle="--", linewidth=2.2, label="Normal fit to latest-period reference")
axis.set(title="Fitted-Normal mapping overlay: full-period score vs reference", xlabel="MarketScore_hat", ylabel="Density", xlim=(0, 100))
axis.grid(alpha=0.2)
axis.legend(frameon=False, fontsize=9)
figure.tight_layout()
quantile_mapping_distribution_plot_path = plots_dir / "normal_quantile_mapping_distribution_overlay_MarketScore_hat.png"
figure.savefig(quantile_mapping_distribution_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved fitted-Normal mapping distribution overlay to: {quantile_mapping_distribution_plot_path}")

In [ ]:
# Compare empirical CDFs with the CDF of the Normal distribution fitted on A.
figure, axis = plt.subplots(figsize=(10, 5.5))
sns.ecdfplot(x=reference_overlay_values, ax=axis, color=REFERENCE_COLOR, linewidth=2.5, label=f"Latest-period reference (n={len(reference_overlay_values):,})")
sns.ecdfplot(x=pre_mapping_score_values, ax=axis, color="#6B7280", linewidth=1.8, linestyle="--", label=f"Before quantile mapping (n={len(pre_mapping_score_values):,})")
sns.ecdfplot(x=mapped_score_values, ax=axis, color=FULL_PERIOD_COLOR, linewidth=2.3, label=f"After fitted-Normal mapping (n={len(mapped_score_values):,})")
axis.plot(score_grid, norm.cdf(score_grid, reference_normal_mean, reference_normal_std), color=REFERENCE_COLOR, linestyle="--", linewidth=2.2, label="Normal fit to latest-period reference")
axis.set(title="Fitted-Normal mapping ECDF overlay: full-period score vs reference", xlabel="MarketScore_hat", ylabel="Cumulative probability", xlim=(0, 100), ylim=(0, 1))
axis.grid(alpha=0.2)
axis.legend(frameon=False)
figure.tight_layout()
quantile_mapping_ecdf_plot_path = plots_dir / "normal_quantile_mapping_ecdf_overlay_MarketScore_hat.png"
figure.savefig(quantile_mapping_ecdf_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved fitted-Normal mapping ECDF overlay to: {quantile_mapping_ecdf_plot_path}")

### A Note

Currently, this is a simple marginal distribution adjustment applied directly to the predicted Market Score. Alternatively, distribution-shift adjustments could be applied to selected explanatory variables before scoring, where economically justified. Because adjusting each predictor independently may distort relationships among market variables, a multivariate approach—such as copula-based mapping—could be considered to preserve the latest-period dependence structure. This would require careful validation to ensure that the adjusted combinations remain realistic and that score calibration, ranking, and temporal stability improve without masking genuine market cycles.

One caution: forcing historical predictors to match the latest-period distribution can remove real economic-cycle information. It should be treated as a sensitivity or contingency method, not automatically as the preferred model.
